#### imports

In [1]:
import utils as ut
import pandas as pd
import os, sys
import numpy as np
import networkx as nx
from datetime import datetime, timedelta

In [2]:
pd.__version__

'2.0.3'

In [3]:
df = ut.read_data("F:\TFG\datasets\\raw_datasets\datalake_v3.csv")

In [4]:
df

,matchId,id_H,id_A,Div,div_order,Date,Weekday,season,HomeTeam,AwayTeam,...,AHW,B365H,B365D,B365A,MaxH,MaxD,MaxA,AvgH,AvgD,AvgA
0,0,8,19,D1,0,2000-08-11,4,T00-01,Dortmund,Hansa Rostock,...,0.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,1,2,20,D1,1,2000-08-12,5,T00-01,Bayern Munich,Hertha,...,0.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,2,15,35,D1,2,2000-08-12,5,T00-01,Freiburg,Stuttgart,...,0.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,3,17,29,D1,3,2000-08-12,5,T00-01,Hamburg,Munich 1860,...,0.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,4,23,4,D1,4,2000-08-12,5,T00-01,Kaiserslautern,Bochum,...,0.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
111410,111410,483,532,T1,9439,2000-05-21,6,T99-00,Ankaragucu,Trabzonspor,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
111411,111411,486,482,T1,9440,2000-05-21,6,T99-00,Antalyaspor,Altay,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
111412,111412,490,515,T1,9441,2000-05-21,6,T99-00,Bursaspor,Kocaelispor,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
111413,111413,497,488,T1,9442,2000-05-21,6,T99-00,Erzurumspor,Besiktas,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [5]:
df.Date.sort_values()

6414     1993-08-07
6415     1993-08-07
6416     1993-08-07
6411     1993-08-07
6410     1993-08-07
            ...    
101962   2024-06-02
101961   2024-06-02
101960   2024-06-02
81380    2024-06-02
101967   2024-06-02
Name: Date, Length: 111415, dtype: datetime64[ns]

In [6]:
COLS_META   = ['matchId','Div','Date','season','id_H','id_A','HomeTeam','AwayTeam']
df[COLS_META]

,matchId,Div,Date,season,id_H,id_A,HomeTeam,AwayTeam
0,0,D1,2000-08-11,T00-01,8,19,Dortmund,Hansa Rostock
1,1,D1,2000-08-12,T00-01,2,20,Bayern Munich,Hertha
2,2,D1,2000-08-12,T00-01,15,35,Freiburg,Stuttgart
3,3,D1,2000-08-12,T00-01,17,29,Hamburg,Munich 1860
4,4,D1,2000-08-12,T00-01,23,4,Kaiserslautern,Bochum
...,...,...,...,...,...,...,...,...
111410,111410,T1,2000-05-21,T99-00,483,532,Ankaragucu,Trabzonspor
111411,111411,T1,2000-05-21,T99-00,486,482,Antalyaspor,Altay
111412,111412,T1,2000-05-21,T99-00,490,515,Bursaspor,Kocaelispor
111413,111413,T1,2000-05-21,T99-00,497,488,Erzurumspor,Besiktas


In [28]:
import torch

In [46]:
def rank_probability_score(logits,actual):
    rps = torch.sum(np.power(logits-actual,2),1)
    rps = rps / logits.shape[1]
    return rps

def avg_rps(logits,actual):
    res = rank_probability_score(logits,actual)
    return float(torch.mean(res,axis=0))

In [47]:
rps = avg_rps(logits,actual)
rps, type(rps)

(0.18779999017715454, float)

# PI RATING

## FUNCTIONS

In [61]:
def get_rate_global(elemH,elemA):
    return (elemH+elemA)/2

def asign_col(df,col_name,value):
    df.loc[:,col_name] = value

## LOGIC

In [62]:
data_initialization = df[df.Date<"1996-07-01"]
old_data_init = data_initialization.copy()
old_data_init

,matchId,id_H,id_A,Div,div_order,Date,season,HomeTeam,AwayTeam,FTHG,...,AHW,B365H,B365D,B365A,MaxH,MaxD,MaxA,AvgH,AvgD,AvgA
6280,6280,2,15,D1,6280,1993-07-08,T93-94,Bayern Munich,Freiburg,3.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
6281,6281,8,24,D1,6281,1993-07-08,T93-94,Dortmund,Karlsruhe,2.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
6282,6282,10,26,D1,6282,1993-07-08,T93-94,Duisburg,Leverkusen,2.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
6283,6283,13,23,D1,6283,1993-07-08,T93-94,FC Koln,Kaiserslautern,0.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
6284,6284,17,30,D1,6284,1993-07-08,T93-94,Hamburg,Nurnberg,5.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
56905,56905,254,242,SP2,9192,1996-01-12,T96-97,Ecija,Ath Madrid B,0.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
56906,56906,271,243,SP2,9193,1996-01-12,T96-97,Las Palmas,Badajoz,2.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
56907,56907,272,255,SP2,9194,1996-01-12,T96-97,Leganes,Eibar,1.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
56908,56908,284,274,SP2,9195,1996-01-12,T96-97,Merida,Levante,1.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [64]:
data_initialization = old_data_init.copy()

cols_pirating = ["matchId","id_H","id_A","HomeTeam","AwayTeam","Div","Date","FTHG","FTAG","FTR"] 
data_initialization = data_initialization[cols_pirating]

asign_col(data_initialization,"FTHG",data_initialization.FTHG.astype(float))
asign_col(data_initialization,"FTAG",data_initialization.FTAG.astype(float))

In [65]:
asign_col(data_initialization,"rate_home",0.0)
asign_col(data_initialization,"rate_away",0.0)
asign_col(data_initialization,"new_rate_home",0.0)
asign_col(data_initialization,"new_rate_away",0.0)
rate_global = get_rate_global(data_initialization.rate_home,data_initialization.rate_away)
asign_col(data_initialization,"rate_global",rate_global)
asign_col(data_initialization,"new_rate_global",0.0)

In [66]:
teams = df.HomeTeam.unique()
print(teams.shape)
dict_rates = { t:{"rate_home":0,"rate_away":0,"rate_global":0} for t in teams }

(275,)


In [67]:
data_initialization

,matchId,id_H,id_A,HomeTeam,AwayTeam,Div,Date,FTHG,FTAG,FTR,rate_home,rate_away,new_rate_home,new_rate_away,rate_global,new_rate_global
6280,6280,2,15,Bayern Munich,Freiburg,D1,1993-07-08,3.0,1.0,H,0.0,0.0,0.0,0.0,0.0,0.0
6281,6281,8,24,Dortmund,Karlsruhe,D1,1993-07-08,2.0,1.0,H,0.0,0.0,0.0,0.0,0.0,0.0
6282,6282,10,26,Duisburg,Leverkusen,D1,1993-07-08,2.0,2.0,D,0.0,0.0,0.0,0.0,0.0,0.0
6283,6283,13,23,FC Koln,Kaiserslautern,D1,1993-07-08,0.0,2.0,A,0.0,0.0,0.0,0.0,0.0,0.0
6284,6284,17,30,Hamburg,Nurnberg,D1,1993-07-08,5.0,2.0,H,0.0,0.0,0.0,0.0,0.0,0.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
56905,56905,254,242,Ecija,Ath Madrid B,SP2,1996-01-12,0.0,1.0,A,0.0,0.0,0.0,0.0,0.0,0.0
56906,56906,271,243,Las Palmas,Badajoz,SP2,1996-01-12,2.0,2.0,D,0.0,0.0,0.0,0.0,0.0,0.0
56907,56907,272,255,Leganes,Eibar,SP2,1996-01-12,1.0,2.0,A,0.0,0.0,0.0,0.0,0.0,0.0
56908,56908,284,274,Merida,Levante,SP2,1996-01-12,1.0,1.0,D,0.0,0.0,0.0,0.0,0.0,0.0


In [68]:
asign_col(data_initialization,"gd_actual", data_initialization.FTHG - data_initialization.FTAG)
asign_col(data_initialization,"gd_pred",0.0)
asign_col(data_initialization,"error",0.0)

In [69]:
data_initialization

,matchId,id_H,id_A,HomeTeam,AwayTeam,Div,Date,FTHG,FTAG,FTR,rate_home,rate_away,new_rate_home,new_rate_away,rate_global,new_rate_global,gd_actual,gd_pred,error
6280,6280,2,15,Bayern Munich,Freiburg,D1,1993-07-08,3.0,1.0,H,0.0,0.0,0.0,0.0,0.0,0.0,2.0,0.0,0.0
6281,6281,8,24,Dortmund,Karlsruhe,D1,1993-07-08,2.0,1.0,H,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0
6282,6282,10,26,Duisburg,Leverkusen,D1,1993-07-08,2.0,2.0,D,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
6283,6283,13,23,FC Koln,Kaiserslautern,D1,1993-07-08,0.0,2.0,A,0.0,0.0,0.0,0.0,0.0,0.0,-2.0,0.0,0.0
6284,6284,17,30,Hamburg,Nurnberg,D1,1993-07-08,5.0,2.0,H,0.0,0.0,0.0,0.0,0.0,0.0,3.0,0.0,0.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
56905,56905,254,242,Ecija,Ath Madrid B,SP2,1996-01-12,0.0,1.0,A,0.0,0.0,0.0,0.0,0.0,0.0,-1.0,0.0,0.0
56906,56906,271,243,Las Palmas,Badajoz,SP2,1996-01-12,2.0,2.0,D,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
56907,56907,272,255,Leganes,Eibar,SP2,1996-01-12,1.0,2.0,A,0.0,0.0,0.0,0.0,0.0,0.0,-1.0,0.0,0.0
56908,56908,284,274,Merida,Levante,SP2,1996-01-12,1.0,1.0,D,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0


In [70]:
b = 10
c = 3
lamda = .01
gamma = 0.2

def error_func(actual,pred):
        error = abs(actual-pred)
        error = c * np.log10(1+error)
        error_H = -error
        error_A = -error
        if actual>pred:
                error_H = error
        if actual<pred: 
                error_A = error
        return error_H, error_A

In [72]:

for row in data_initialization.itertuples():
    idx = row.Index
    team_H = row.HomeTeam
    team_A = row.AwayTeam
    data_initialization.loc[idx,"rate_home"] = dict_rates[team_H]["rate_home"]
    data_initialization.loc[idx,"rate_away"] = dict_rates[team_A]["rate_away"]
    gd_H_pred = b ** (abs(dict_rates[team_H]["rate_home"])/c) - 1
    gd_A_pred = b ** (abs(dict_rates[team_A]["rate_away"])/c) - 1
    data_initialization.loc[idx,"gd_pred"] = gd_H_pred - gd_A_pred
    error_H, error_A = error_func(row.gd_actual,row.gd_pred)
    data_initialization.loc[idx,"error"] = abs(error_H)

    new_rate_home_H = dict_rates[team_H]["rate_home"] + error_H * lamda
    new_rate_home_A = dict_rates[team_H]["rate_away"] + (new_rate_home_H - dict_rates[team_H]["rate_home"]) * gamma
    new_rate_away_A = dict_rates[team_A]["rate_away"] + error_A * lamda
    new_rate_away_H = dict_rates[team_A]["rate_home"] + (new_rate_away_A - dict_rates[team_A]["rate_away"]) * gamma

    dict_rates[team_H]["rate_home"] = new_rate_home_H
    dict_rates[team_A]["rate_home"] = new_rate_away_H
    dict_rates[team_A]["rate_global"] = (new_rate_home_H + new_rate_away_H) / 2
    dict_rates[team_A]["rate_away"] = new_rate_away_A
    dict_rates[team_A]["rate_home"] = new_rate_away_H
    dict_rates[team_A]["rate_global"] = (new_rate_away_H + new_rate_away_A) / 2
        
    data_initialization.loc[idx,"new_rate_home"] = new_rate_home_H
    data_initialization.loc[idx,"new_rate_away"] = new_rate_away_A
    

In [73]:
data_initialization

,matchId,id_H,id_A,HomeTeam,AwayTeam,Div,Date,FTHG,FTAG,FTR,rate_home,rate_away,new_rate_home,new_rate_away,rate_global,new_rate_global,gd_actual,gd_pred,error
6280,6280,2,15,Bayern Munich,Freiburg,D1,1993-07-08,3.0,1.0,H,0.000000,0.000000,0.014314,-0.014314,0.0,0.0,2.0,0.000000,1.431364
6281,6281,8,24,Dortmund,Karlsruhe,D1,1993-07-08,2.0,1.0,H,0.000000,0.000000,0.009031,-0.009031,0.0,0.0,1.0,0.000000,0.903090
6282,6282,10,26,Duisburg,Leverkusen,D1,1993-07-08,2.0,2.0,D,0.000000,0.000000,0.000000,0.000000,0.0,0.0,0.0,0.000000,0.000000
6283,6283,13,23,FC Koln,Kaiserslautern,D1,1993-07-08,0.0,2.0,A,0.000000,0.000000,-0.014314,0.014314,0.0,0.0,-2.0,0.000000,1.431364
6284,6284,17,30,Hamburg,Nurnberg,D1,1993-07-08,5.0,2.0,H,0.000000,0.000000,0.018062,-0.018062,0.0,0.0,3.0,0.000000,1.806180
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
56905,56905,254,242,Ecija,Ath Madrid B,SP2,1996-01-12,0.0,1.0,A,0.010120,-0.018062,0.001089,-0.009031,0.0,0.0,-1.0,-0.006162,0.903090
56906,56906,271,243,Las Palmas,Badajoz,SP2,1996-01-12,2.0,2.0,D,0.009031,0.014314,0.009031,0.014314,0.0,0.0,0.0,-0.004091,0.000000
56907,56907,272,255,Leganes,Eibar,SP2,1996-01-12,1.0,2.0,A,0.000000,0.000000,-0.009031,0.009031,0.0,0.0,-1.0,0.000000,0.903090
56908,56908,284,274,Merida,Levante,SP2,1996-01-12,1.0,1.0,D,0.007131,-0.009031,0.007131,-0.009031,0.0,0.0,0.0,-0.001468,0.000000


In [76]:
final_rates = pd.DataFrame(dict_rates).T.reset_index()
teams = df[['HomeTeam','AwayTeam']].drop_duplicates()
final_rates.merge(teams, left_on='index', right_on='HomeTeam', how='left').sort_values("rate_global",ascending=False)

,index,rate_home,rate_away,rate_global,HomeTeam,AwayTeam
1654,Man United,0.611902,0.247909,0.434421,Man United,Hull
1645,Man United,0.611902,0.247909,0.434421,Man United,Birmingham
1647,Man United,0.611902,0.247909,0.434421,Man United,Portsmouth
1649,Man United,0.611902,0.247909,0.434421,Man United,Norwich
1650,Man United,0.611902,0.247909,0.434421,Man United,Crystal Palace
...,...,...,...,...,...,...
1770,Ipswich,-0.215195,-0.334690,-0.274943,Ipswich,Crystal Palace
1769,Ipswich,-0.215195,-0.334690,-0.274943,Ipswich,Nott'm Forest
1768,Ipswich,-0.215195,-0.334690,-0.274943,Ipswich,Swindon
1767,Ipswich,-0.215195,-0.334690,-0.274943,Ipswich,QPR


# PAGE RANK

In [9]:
old_data = df.copy()
# data = data[data.Div=='SP1']
# ut.getPoints(data,"FTHG","FTAG","Points_H")
# ut.getPoints(data,"FTAG","FTHG","Points_A")
df

,matchId,id_H,id_A,Div,div_order,Date,season,HomeTeam,AwayTeam,FTHG,...,AHW,B365H,B365D,B365A,MaxH,MaxD,MaxA,AvgH,AvgD,AvgA
0,0,8,19,D1,0,2000-08-11,T00-01,Dortmund,Hansa Rostock,1,...,0.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,1,2,20,D1,1,2000-08-12,T00-01,Bayern Munich,Hertha,4,...,0.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,2,15,35,D1,2,2000-08-12,T00-01,Freiburg,Stuttgart,4,...,0.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,3,17,29,D1,3,2000-08-12,T00-01,Hamburg,Munich 1860,2,...,0.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,4,23,4,D1,4,2000-08-12,T00-01,Kaiserslautern,Bochum,0,...,0.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
58541,58541,258,243,SP2,10828,2000-06-04,T99-00,Extremadura,Badajoz,2.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
58542,58542,271,274,SP2,10829,2000-06-04,T99-00,Las Palmas,Levante,2.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
58543,58543,277,314,SP2,10830,2000-06-04,T99-00,Logrones,Villarreal,1.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
58544,58544,288,297,SP2,10831,2000-06-04,T99-00,Osasuna,Recreativo,2.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [10]:
def get_derby_id(data,columns):
    unique_ids = np.sort(data[columns].values,axis=1)
    df_derby = pd.DataFrame(np.unique(unique_ids,axis=0),columns=["id1","id2"])
    df_derby.reset_index(inplace=True,names='derby')
    return df_derby, unique_ids

def add_column_from_df(df,other_df,list_values,columns):
    for i,c in enumerate(columns):
        df.loc[:,c] = list_values[:,i]
    df = df.merge(other_df,on=["id1","id2"])
    return df

In [282]:
match_sorted_list, match_sorted = get_derby_id(df,["id_H","id_A"])
df = add_column_from_df(df,match_sorted_list,match_sorted,["id1","id2"])
df

<ipython-input-281-08fd3addd4b9>:9: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df.loc[:,c] = list_values[:,i]
<ipython-input-281-08fd3addd4b9>:9: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df.loc[:,c] = list_values[:,i]


,matchId,id_H,id_A,Div,div_order,Date,season,HomeTeam,AwayTeam,FTHG,...,MaxD,MaxA,AvgH,AvgD,AvgA,Points_H,Points_A,id1,id2,derby
0,37021,190,212,SP1,0,2000-09-09,T00-01,Barcelona,Malaga,2,...,NaN,NaN,NaN,NaN,NaN,3,0,190,212,206
1,37213,212,190,SP1,192,2001-01-27,T00-01,Malaga,Barcelona,0,...,NaN,NaN,NaN,NaN,NaN,1,1,190,212,206
2,37484,212,190,SP1,463,2001-10-20,T01-02,Malaga,Barcelona,1,...,NaN,NaN,NaN,NaN,NaN,1,1,190,212,206
3,37676,190,212,SP1,655,2002-03-03,T01-02,Barcelona,Malaga,5,...,NaN,NaN,NaN,NaN,NaN,3,0,190,212,206
4,37947,212,190,SP1,926,2003-12-01,T02-03,Malaga,Barcelona,0,...,NaN,NaN,NaN,NaN,NaN,1,1,190,212,206
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
10687,47290,221,185,SP1,10269,1999-05-16,T98-99,Salamanca,Alaves,1,...,NaN,NaN,NaN,NaN,NaN,3,0,185,221,25
10688,47107,199,213,SP1,10086,1999-03-01,T98-99,Extremadura,Mallorca,1,...,NaN,NaN,NaN,NaN,NaN,3,0,199,213,428
10689,47297,213,199,SP1,10276,1999-05-23,T98-99,Mallorca,Extremadura,2,...,NaN,NaN,NaN,NaN,NaN,3,0,199,213,428
10690,47142,230,199,SP1,10121,1999-01-24,T98-99,Villarreal,Extremadura,1,...,NaN,NaN,NaN,NaN,NaN,1,1,199,230,440


In [283]:
points_H = df.melt(id_vars=["derby","id_H","Date"],value_vars=["Points_H"])
points_A = df.melt(id_vars=["derby","id_A","Date"],value_vars=["Points_A"])
points_H.columns = points_A.columns = ["derby","id","Date","variable","value"]
points_df = pd.concat([points_H,points_A]) 
points_df

,derby,id,Date,variable,value
0,206,190,2000-09-09,Points_H,3
1,206,212,2001-01-27,Points_H,1
2,206,212,2001-10-20,Points_H,1
3,206,190,2002-03-03,Points_H,3
4,206,212,2003-12-01,Points_H,1
...,...,...,...,...,...
10687,25,185,1999-05-16,Points_A,0
10688,428,213,1999-03-01,Points_A,0
10689,428,199,1999-05-23,Points_A,0
10690,440,199,1999-01-24,Points_A,1


In [284]:
page_rank_matrix = ut.compute_lag(points_df,lag=5,cols_group=["derby","id"],col_window="Date",min_samples=2,aggregations={"value":"mean"})
page_rank_matrix = page_rank_matrix.reset_index().sort_values("Date").dropna()
page_rank_matrix

,derby,id,Date,value_5
19333,769,225,1993-03-10,1.000000
14421,557,224,1993-03-10,1.600000
4763,171,219,1993-03-10,1.800000
9770,317,193,1993-03-10,1.800000
1159,62,186,1993-03-10,2.000000
...,...,...,...,...
19870,780,227,2021-12-05,1.200000
19796,780,223,2021-12-05,1.800000
2351,114,188,2021-12-05,2.333333
5042,176,224,2021-12-05,0.200000


In [11]:
def get_between(df,date1,date2):
    mask_date1 = df.Date>date1
    mask_date2 = df.Date<=date2
    return df[mask_date1 & mask_date2]

In [286]:
# idxs = page_rank_matrix[page_rank_matrix.Date<="2000-04-06"]
df_filtered = get_between(page_rank_matrix,"2008-07-01","2011-07-01")
idxs = df_filtered[["derby","id"]].drop_duplicates(keep='last').index
idxs

Index([ 2697, 17223,  7908, 17284,  1502, 17163,  8346,  4568,  6301, 15862,
       ...
       20690, 20381,  4171,  1857, 20698, 12183,  4155, 17400, 17418, 12123],
      dtype='int64', length=540)

In [287]:
page_rank_matrix_filt = page_rank_matrix.loc[idxs].drop(columns='Date')
# page_rank_matrix_filt = page_rank_matrix_filt.set_index("derby").sort_index()
page_rank_matrix_filt = page_rank_matrix_filt.sort_values("derby")
page_rank_matrix_filt

,derby,id,value_5
1225,66,188,2.000000
1207,66,187,1.333333
1243,67,189,2.333333
1237,67,187,0.800000
1255,68,187,0.666667
...,...,...,...
21170,812,228,2.600000
21220,814,228,2.250000
21259,814,232,0.800000
21344,818,230,2.600000


In [288]:
page_rank_matrix_filt_1 = page_rank_matrix_filt[::2]
page_rank_matrix_filt_2 = page_rank_matrix_filt[1::2]
page_rank_matrix_pivoted = page_rank_matrix_filt_1.merge(page_rank_matrix_filt_2, on='derby', suffixes=('_1','_2'))
page_rank_matrix_pivoted

,derby,id_1,value_5_1,id_2,value_5_2
0,66,188,2.000000,187,1.333333
1,67,189,2.333333,187,0.800000
2,68,187,0.666667,190,2.200000
3,69,191,1.750000,187,1.750000
4,74,187,1.000000,198,1.800000
...,...,...,...,...,...
259,808,230,2.400000,227,0.600000
260,810,232,0.200000,227,2.600000
261,812,230,0.200000,228,2.600000
262,814,228,2.250000,232,0.800000


In [52]:
team_id_name = df[["id_H","HomeTeam"]].set_index("id_H")["HomeTeam"].to_dict()

In [53]:
page_rank_matrix_pivoted.loc[:,"HomeTeam"] = page_rank_matrix_pivoted.id_1.map(team_id_name)
page_rank_matrix_pivoted.loc[:,"AwayTeam"] = page_rank_matrix_pivoted.id_2.map(team_id_name)

In [54]:
page_rank_matrix_pivoted.HomeTeam.unique().shape

(26,)

In [62]:
teams = page_rank_matrix_pivoted.HomeTeam.unique()
matrix_page_rank = pd.DataFrame(columns=teams,index=teams)

for row in page_rank_matrix_pivoted.itertuples():
    matrix_page_rank.loc[row.HomeTeam,row.AwayTeam] = row.value_5_1
    matrix_page_rank.loc[row.AwayTeam,row.HomeTeam] = row.value_5_2

In [63]:
dict_team_page = { t:i for t,i in enumerate(matrix_page_rank.iloc[0].index)}
dict_team_page

{0: 'Ath Bilbao',
 1: 'Ath Madrid',
 2: 'Almeria',
 3: 'Betis',
 4: 'Getafe',
 5: 'La Coruna',
 6: 'Levante',
 7: 'Malaga',
 8: 'Mallorca',
 9: 'Osasuna',
 10: 'Sevilla',
 11: 'Sp Gijon',
 12: 'Valencia',
 13: 'Valladolid',
 14: 'Hercules',
 15: 'Numancia',
 16: 'Recreativo',
 17: 'Tenerife',
 18: 'Espanol',
 19: 'Santander',
 20: 'Zaragoza',
 21: 'Barcelona',
 22: 'Real Madrid',
 23: 'Sociedad',
 24: 'Villarreal',
 25: 'Vallecano'}

In [64]:
# matrix_page_rank = matrix_page_rank.dropna().drop(columns='NaN')
matrix_page_rank = matrix_page_rank.sort_index()
matrix_page_rank = matrix_page_rank[sorted(matrix_page_rank.columns)]
matrix_page_rank

,Almeria,Ath Bilbao,Ath Madrid,Barcelona,Betis,Espanol,Getafe,Hercules,La Coruna,Levante,...,Santander,Sevilla,Sociedad,Sp Gijon,Tenerife,Valencia,Valladolid,Vallecano,Villarreal,Zaragoza
Almeria,NaN,1.333333,0.8,0.666667,1.75,1.0,2.0,NaN,1.333333,0.8,...,0.666667,1.0,1.666667,1.4,NaN,0.333333,1.4,NaN,0.6,1.0
Ath Bilbao,2.0,NaN,1.2,0.4,1.6,0.2,0.6,3.0,1.0,2.0,...,1.4,0.6,2.6,1.5,3.0,1.8,1.4,NaN,0.8,1.2
Ath Madrid,2.333333,1.8,NaN,0.8,1.8,1.4,1.6,NaN,1.2,3.0,...,2.2,1.8,0.0,2.6,2.0,0.4,0.8,NaN,1.4,1.0
Barcelona,2.2,2.2,2.0,NaN,1.0,1.6,2.2,0.0,3.0,3.0,...,2.6,1.4,1.6,2.6,2.2,1.0,2.2,NaN,1.8,2.6
Betis,1.75,1.0,1.2,1.6,NaN,1.6,2.0,NaN,1.2,1.5,...,1.6,0.2,NaN,1.4,NaN,0.8,2.0,NaN,0.8,NaN
Espanol,1.8,2.6,1.4,1.0,1.0,NaN,1.6,NaN,1.8,1.6,...,0.8,1.8,1.6,1.0,2.6,0.6,1.0,NaN,1.2,1.0
Getafe,1.4,1.8,1.0,0.4,1.8,1.0,NaN,NaN,0.8,1.6,...,1.4,1.8,1.0,2.0,NaN,2.0,1.5,NaN,0.2,1.6
Hercules,NaN,1.5,NaN,1.5,NaN,NaN,NaN,NaN,1.5,NaN,...,NaN,3.0,3.0,1.0,NaN,0.0,NaN,NaN,NaN,2.0
La Coruna,0.8,1.6,1.8,0.0,1.2,1.2,2.0,3.0,NaN,2.0,...,1.6,1.4,1.6,0.5,1.6,0.4,2.6,NaN,1.8,0.8
Levante,2.4,1.333333,0.4,0.2,2.4,0.666667,1.666667,NaN,0.8,NaN,...,2.333333,0.2,1.0,1.666667,NaN,0.8,NaN,NaN,0.8,2.0


In [58]:
df[df.id_H==222]

,matchId,id_H,id_A,Div,div_order,Date,season,HomeTeam,AwayTeam,FTHG,...,MaxD,MaxA,AvgH,AvgD,AvgA,Points_H,Points_A,id1,id2,derby
135,37218,222,224,SP1,197,2001-01-28,T00-01,Santander,Sociedad,1,...,NaN,NaN,NaN,NaN,NaN,0,3,222,224,768
137,38032,222,224,SP1,1011,2003-03-15,T02-03,Santander,Sociedad,1,...,NaN,NaN,NaN,NaN,NaN,0,3,222,224,768
138,38183,222,224,SP1,1162,2003-09-13,T03-04,Santander,Sociedad,0,...,NaN,NaN,NaN,NaN,NaN,0,3,222,224,768
141,38946,222,224,SP1,1925,2005-04-12,T05-06,Santander,Sociedad,2,...,NaN,NaN,NaN,NaN,NaN,1,1,222,224,768
143,39366,222,224,SP1,2345,2007-01-14,T06-07,Santander,Sociedad,1,...,NaN,NaN,NaN,NaN,NaN,3,0,222,224,768
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
10556,45841,222,221,SP1,8820,1995-12-20,T95-96,Santander,Salamanca,2.0,...,NaN,NaN,NaN,NaN,NaN,3,0,221,222,757
10558,46580,222,221,SP1,9559,1997-08-31,T97-98,Santander,Salamanca,1,...,NaN,NaN,NaN,NaN,NaN,3,0,221,222,757
10560,47049,222,221,SP1,10028,1998-11-15,T98-99,Santander,Salamanca,4,...,NaN,NaN,NaN,NaN,NaN,3,0,221,222,757
10603,46401,222,199,SP1,9380,1997-02-03,T96-97,Santander,Extremadura,2.0,...,NaN,NaN,NaN,NaN,NaN,0,3,199,222,432


In [67]:
page_rank_matrix_pivoted

,derby,id_1,value_5_1,id_2,value_5_2,HomeTeam,AwayTeam
0,66,188,2.000000,187,1.333333,Ath Bilbao,Almeria
1,67,189,2.333333,187,0.800000,Ath Madrid,Almeria
2,68,187,0.666667,190,2.200000,Almeria,Barcelona
3,69,191,1.750000,187,1.750000,Betis,Almeria
4,74,187,1.000000,198,1.800000,Almeria,Espanol
...,...,...,...,...,...,...,...
259,808,230,2.400000,227,0.600000,Villarreal,Valencia
260,810,232,0.200000,227,2.600000,Zaragoza,Valencia
261,812,230,0.200000,228,2.600000,Villarreal,Valladolid
262,814,228,2.250000,232,0.800000,Valladolid,Zaragoza


In [81]:
import pandas as pd
import networkx as nx

# Create an empty directed graph
graph = nx.DiGraph()

# Iterate over the DataFrame rows
for _, row in page_rank_matrix_pivoted.iterrows():
    source = row['HomeTeam']
    target = row['AwayTeam']
    weight = row['value_5_2']
    
    # Add edges only for non-NaN values
    if not pd.isna(weight):
        graph.add_edge(source, target, weight=weight)

    source = row['AwayTeam']
    target = row['HomeTeam']
    weight = row['value_5_1']
    
    # Add edges only for non-NaN values
    if not pd.isna(weight):
        graph.add_edge(source, target, weight=weight)

# Print the graph
print(graph.edges(data=True))


[('Ath Bilbao', 'Almeria', {'weight': 1.3333333333333333}), ('Ath Bilbao', 'Ath Madrid', {'weight': 1.8}), ('Ath Bilbao', 'Barcelona', {'weight': 2.2}), ('Ath Bilbao', 'Betis', {'weight': 1.0}), ('Ath Bilbao', 'Espanol', {'weight': 2.6}), ('Ath Bilbao', 'Getafe', {'weight': 1.8}), ('Ath Bilbao', 'Hercules', {'weight': 1.5}), ('Ath Bilbao', 'La Coruna', {'weight': 1.6}), ('Ath Bilbao', 'Levante', {'weight': 1.3333333333333333}), ('Ath Bilbao', 'Malaga', {'weight': 0.8}), ('Ath Bilbao', 'Mallorca', {'weight': 1.6}), ('Ath Bilbao', 'Numancia', {'weight': 1.0}), ('Ath Bilbao', 'Osasuna', {'weight': 2.2}), ('Ath Bilbao', 'Real Madrid', {'weight': 3.0}), ('Ath Bilbao', 'Recreativo', {'weight': 1.0}), ('Ath Bilbao', 'Santander', {'weight': 1.4}), ('Ath Bilbao', 'Sevilla', {'weight': 2.4}), ('Ath Bilbao', 'Sociedad', {'weight': 0.2}), ('Ath Bilbao', 'Sp Gijon', {'weight': 0.6}), ('Ath Bilbao', 'Tenerife', {'weight': 1.4}), ('Ath Bilbao', 'Valencia', {'weight': 1.2}), ('Ath Bilbao', 'Valladolid

In [82]:
import networkx as nx
import numpy as np

# Assuming you have a matrix called 'adj_matrix' representing the adjacency matrix of the graph
# You can convert the matrix to a NetworkX graph using the 'from_numpy_array' function
# graph = nx.from_numpy_array(matrix_page_rank.T.fillna(-1).values)

# Calculate the PageRank scores using the 'pagerank' function
pagerank_scores_1 = nx.pagerank(graph,max_iter=100, tol=1e-5)

# Access the PageRank score for each node
for node, score in pagerank_scores_1.items():
    print(f"Node {node}: PageRank score = {score}")

Node Ath Bilbao: PageRank score = 0.04037090540758752
Node Almeria: PageRank score = 0.031931199146513586
Node Ath Madrid: PageRank score = 0.03889193045618663
Node Barcelona: PageRank score = 0.057600983655435624
Node Betis: PageRank score = 0.03401371436092574
Node Espanol: PageRank score = 0.042706364550805495
Node Getafe: PageRank score = 0.03807781483647695
Node La Coruna: PageRank score = 0.0476556894423996
Node Levante: PageRank score = 0.032408135989923675
Node Malaga: PageRank score = 0.03638183108784246
Node Mallorca: PageRank score = 0.03921600116396907
Node Osasuna: PageRank score = 0.037324268258363456
Node Real Madrid: PageRank score = 0.05716140490043884
Node Recreativo: PageRank score = 0.02360484787123895
Node Santander: PageRank score = 0.044080988484003734
Node Sevilla: PageRank score = 0.052816924868864586
Node Sociedad: PageRank score = 0.04028760869470284
Node Sp Gijon: PageRank score = 0.03476501343392258
Node Valencia: PageRank score = 0.05077282429520663
Node V

In [105]:
pagerank_scores_1

{'Ath Bilbao': 0.04037090540758752,
 'Almeria': 0.031931199146513586,
 'Ath Madrid': 0.03889193045618663,
 'Barcelona': 0.057600983655435624,
 'Betis': 0.03401371436092574,
 'Espanol': 0.042706364550805495,
 'Getafe': 0.03807781483647695,
 'La Coruna': 0.0476556894423996,
 'Levante': 0.032408135989923675,
 'Malaga': 0.03638183108784246,
 'Mallorca': 0.03921600116396907,
 'Osasuna': 0.037324268258363456,
 'Real Madrid': 0.05716140490043884,
 'Recreativo': 0.02360484787123895,
 'Santander': 0.044080988484003734,
 'Sevilla': 0.052816924868864586,
 'Sociedad': 0.04028760869470284,
 'Sp Gijon': 0.03476501343392258,
 'Valencia': 0.05077282429520663,
 'Valladolid': 0.042152388027313435,
 'Villarreal': 0.04753711587863731,
 'Zaragoza': 0.028910237381441384,
 'Hercules': 0.023215814037299044,
 'Numancia': 0.031722340408664965,
 'Tenerife': 0.037455970172823434,
 'Vallecano': 0.008937683189012438}

In [83]:
pd.DataFrame(pagerank_scores_1.values(), pagerank_scores_1.keys(), columns=['page_rank']).sort_values("page_rank")

,page_rank
Vallecano,0.008938
Hercules,0.023216
Recreativo,0.023605
Zaragoza,0.028910
Numancia,0.031722
Almeria,0.031931
Levante,0.032408
Betis,0.034014
Sp Gijon,0.034765
Malaga,0.036382


## PageRank function

In [64]:
# old_data = data.copy()
df.shape

(60005, 39)

In [200]:
# FUNCIONES AUXILIARES 

def get_derby_id(data,columns):
    unique_ids = np.sort(data[columns].values,axis=1).astype(str)
    df_derby = pd.DataFrame(np.unique(unique_ids,axis=0),columns=["id1","id2"])
    df_derby.reset_index(inplace=True,names='derby')
    return df_derby, unique_ids

def add_column_from_df(df,other_df,list_values,columns):
    for i,c in enumerate(columns):
        df.loc[:,c] = list_values[:,i]
    df = df.merge(other_df,on=["id1","id2"])
    return df

def get_between(df,date1,date2):
    mask_date1 = df.Date>date1
    mask_date2 = df.Date<=date2
    return df[mask_date1 & mask_date2]

In [201]:
def create_graph(data,home='HomeTeam',away='AwayTeam'):
    graph = nx.DiGraph()

    # Iterate over the DataFrame rows
    for _, row in data.iterrows():
        source = row[home]
        target = row[away]
        weight = row['value_2']
        
        # Add edges only for non-NaN values
        if not pd.isna(weight):
            graph.add_edge(source, target, weight=weight)

        source = row[away]
        target = row[home]
        weight = row['value_1']
        
        # Add edges only for non-NaN values
        if not pd.isna(weight):
            graph.add_edge(source, target, weight=weight)

    return graph

def create_graph_mapping_teams(data):
    data.loc[:,"home_div"] = data.HomeTeam + '_' + data.Div
    data.loc[:,"away_div"] = data.AwayTeam + '_' + data.Div
    graph = create_graph(data,"home_div","away_div")
    return graph



In [202]:
initial_date = "1996-07-01"

def get_points_lags(df,date,lag=5,div=None,delta=2,min_samples=1) -> pd.DataFrame:
    if div is not None: df = df[df.Div==div]
    ut.getPoints(df,"FTHG","FTAG","Points_H")
    ut.getPoints(df,"FTAG","FTHG","Points_A")

    match_sorted_list, match_sorted = get_derby_id(df,["HomeTeam","AwayTeam"])
    df = add_column_from_df(df,match_sorted_list,match_sorted,["id1","id2"])

    points_H = df.melt(id_vars=["Div","derby","HomeTeam","Date"],value_vars=["Points_H"])
    points_A = df.melt(id_vars=["Div","derby","AwayTeam","Date"],value_vars=["Points_A"])
    points_H.columns = points_A.columns = ["Div","derby","id","Date","variable","value"]
    points_df = pd.concat([points_H,points_A]).sort_values("Date")

    page_rank_matrix = ut.compute_lag(points_df,lag=lag,cols_group=["derby","id"],col_window="Date",
                                      min_samples=min_samples,aggregations={"value":"mean"},
                                      cols_agg=["value"],keep=["Div"])
    page_rank_matrix = page_rank_matrix.reset_index().sort_values("Date").dropna()

    initial_date = datetime.strptime(date, "%Y-%m-%d") - timedelta(days=(delta+5)*30)
    matrix_filtered = get_between(page_rank_matrix,initial_date,date)

    return matrix_filtered


In [203]:
from sklearn.preprocessing import maxabs_scale as sc

team_id_name = df[["id_H","HomeTeam"]].drop_duplicates().set_index("id_H")["HomeTeam"].to_dict()

def pagerank(df: pd.DataFrame) -> dict:
    # a nivel de lag no tenemos en cuenta la Division ya que queremos mantener el lag a nivel derby
    idxs = df[["derby","id"]].drop_duplicates(keep='last').index
    if len(idxs)==0: return

    # realizamos una serie de transformaciones para preparar la matriz de entrada del algoritmo de pagerank
    page_rank_matrix_filt = df.loc[idxs].drop(columns='Date')
    page_rank_matrix_filt = page_rank_matrix_filt.sort_values("derby")

    page_rank_matrix_filt_1 = page_rank_matrix_filt[::2]
    page_rank_matrix_filt_2 = page_rank_matrix_filt[1::2]
    page_rank_matrix_pivoted = page_rank_matrix_filt_1.merge(page_rank_matrix_filt_2, on=['derby','Div'], suffixes=('_1','_2'))

    page_rank_matrix_pivoted.loc[:,"HomeTeam"] = page_rank_matrix_pivoted.id_1.map(team_id_name)
    page_rank_matrix_pivoted.loc[:,"AwayTeam"] = page_rank_matrix_pivoted.id_2.map(team_id_name)

    # cracion grafo y calculo pagerank
    graph = create_graph_mapping_teams(page_rank_matrix_pivoted)
    pagerank_scores = nx.pagerank(graph,max_iter=100, tol=1e-5)
    pagerank_scores = pd.DataFrame(pagerank_scores.values(),pagerank_scores.keys(),columns=["pagerank"]).reset_index()
    pagerank_scores.loc[:,"pagerank"] = sc(pagerank_scores['pagerank'],axis=0)
    pagerank_scores[["team","Div"]] = pagerank_scores['index'].str.split("_",expand=True)

    return pagerank_scores.sort_values("pagerank").set_index("team",drop=True).drop(columns=['index'])

Basicamente lo que se hace es se calcula el pagerank de cada partido con un lag de N partidos (por defecto 5 partidos). Posteriormente se crea una lista de fechas comun a todos los equipos, y se busca la fecha de partido mas proxima a cada fecha comun.

In [204]:
def create_date_list(date1, date2, delta=2):
    """
    delta son los meses de padding entre un registro de pagerank y el siguiente
    """
    # Convert the input strings to datetime objects
    date1 = datetime.strptime(date1, "%Y-%m-%d")
    date2 = datetime.strptime(date2, "%Y-%m-%d")
    # Initialize the list of dates
    date_list = []
    # Start with the initial date
    current_date = date1
    # Add the initial date to the list
    date_list.append(current_date.strftime("%Y-%m-%d"))
    # Increment the date by 2 months until reaching the end date
    while current_date < date2:
        # Add a time delta of 2 months to the current date
        current_date += timedelta(days=delta*30)
        
        # Add the updated date to the list
        date_list.append(current_date.strftime("%Y-%m-%d"))
    return date_list

In [205]:
import importlib as il
il.reload(ut)

<module 'utils' from 'f:\\TFG\\code\\old_stuff\\experiments\\utils.py'>

In [206]:
gpl_ej = get_points_lags(df,"2004-12-30",lag=10,delta=4,min_samples=0)

In [209]:
gpl_ej[gpl_ej.id=='Levante']

,derby,id,Date,value,Div
1540512,3100,Levante,2004-04-11,1.8,SP1
1540511,3100,Levante,2004-04-11,1.8,SP1
1540513,3100,Levante,2004-04-11,1.8,SP1
1540514,3100,Levante,2004-04-11,1.8,SP1
1540516,3100,Levante,2004-04-11,1.8,SP1
...,...,...,...,...,...
496602,929,Levante,2004-12-21,0.9,SP1
496603,929,Levante,2004-12-21,0.9,SP1
496616,929,Levante,2004-12-21,0.9,SP1
496617,929,Levante,2004-12-21,0.9,SP1


In [211]:
df_ej = get_points_lags(df,"2004-12-30",lag=10,delta=4,min_samples=0)
print(df_ej.columns)

idxs = df_ej[["derby","id"]].drop_duplicates(keep='last').index
assert len(idxs)!=0

# realizamos una serie de transformaciones para preparar la matriz de entrada del algoritmo de pagerank
page_rank_matrix_filt = df_ej.loc[idxs].drop(columns='Date')
page_rank_matrix_filt = page_rank_matrix_filt.sort_values("derby")

page_rank_matrix_filt_1 = page_rank_matrix_filt[::2]
page_rank_matrix_filt_2 = page_rank_matrix_filt[1::2]
page_rank_matrix_pivoted = page_rank_matrix_filt_1.merge(page_rank_matrix_filt_2, on=['derby','Div'], suffixes=('_1','_2'))

page_rank_matrix_pivoted.loc[:,"HomeTeam"] = page_rank_matrix_pivoted.id_1#.map(team_id_name)
page_rank_matrix_pivoted.loc[:,"AwayTeam"] = page_rank_matrix_pivoted.id_2#.map(team_id_name)

page_rank_matrix_pivoted


Index(['derby', 'id', 'Date', 'value', 'Div'], dtype='object')


,derby,id_1,value_1,Div,id_2,value_2,HomeTeam,AwayTeam
0,17,Ajaccio,1.857143,F1,Auxerre,2.000000,Ajaccio,Auxerre
1,18,Ajaccio,1.857143,F1,Bastia,0.500000,Ajaccio,Bastia
2,19,Ajaccio,0.888889,F1,Bordeaux,3.000000,Ajaccio,Bordeaux
3,24,Guingamp,3.000000,F1,Ajaccio,0.000000,Guingamp,Ajaccio
4,28,Lens,2.000000,F1,Ajaccio,1.166667,Lens,Ajaccio
...,...,...,...,...,...,...,...,...
896,5323,Valencia,2.250000,SP1,Villarreal,1.600000,Valencia,Villarreal
897,5325,Valencia,3.000000,SP1,Zaragoza,1.100000,Valencia,Zaragoza
898,5328,Valladolid,2.333333,SP1,Villarreal,1.600000,Valladolid,Villarreal
899,5341,Zaragoza,1.500000,SP1,Villarreal,1.666667,Zaragoza,Villarreal


In [212]:
page_rank_matrix_pivoted[page_rank_matrix_pivoted.HomeTeam=="Levante"]

,derby,id_1,value_1,Div,id_2,value_2,HomeTeam,AwayTeam
26,103,Levante,0.300,SP1,Alaves,2.5,Levante,Alaves
399,2228,Levante,2.250,SP2,Cordoba,1.4,Levante,Cordoba
519,3100,Levante,1.800,SP1,Getafe,1.4,Levante,Getafe
653,4041,Levante,0.625,SP2,Xerez,1.5,Levante,Xerez


In [71]:
def compute_pagerank(df,delta,lag,dates,min_samples) -> pd.DataFrame:
    """
    Calculamos pagerank para cada slot de fechas y lo concatenamos en un nuevo dataframe
    """
    page_rank_permonth = pd.DataFrame(columns=["Div","Team","pagerank","Date_pagerank"])

    for date in dates:
        print(date,end='\r')
        matrix_lags = get_points_lags(df,date,lag=lag,delta=delta,min_samples=min_samples)
        p_rank = pagerank(matrix_lags)
        if type(p_rank)==pd.DataFrame:
            p_rank.loc[:,"Date_pagerank"] = datetime.strptime(date,"%Y-%m-%d")
            p_rank.loc[:,"Team"] = p_rank.index
            p_rank.loc[:,"Div"] = p_rank.Div 
            p_rank = p_rank.reset_index(drop=True)
            page_rank_permonth = pd.concat([page_rank_permonth,p_rank])

    return page_rank_permonth  

In [72]:
def get_prior_date(date_list, D):
    # Filter the dates in the list that are prior to D
    prior_dates = [date for date in date_list if datetime.strptime(date, "%Y-%m-%d") < D]

    # Find the date in the prior_dates list that is closest to D
    closest_date = min(prior_dates, key=lambda date: (D - datetime.strptime(date, "%Y-%m-%d")).days)

    return closest_date

Añadimos la fecha mas proxima al partido de las creadas (comunes a todos los partidos) para posteriormente hacer el cruce de Pagerank con los datos origen. 

In [73]:
lags = [1,2,5,10]
deltas = [1,1,2,4]

# Define the start and end dates as strings in the format "YYYY-MM-DD"
date1 = "1993-07-01"
date2 = "2023-01-01"

Calculamos el pagerank para cada pareja de lag-deltam

In [75]:
lags = [1,2,5,10]
deltas = [1,1,2,4]
min_samples=0

pagerank_final = pd.DataFrame

for delta,lag in zip(deltas,lags):
    dates = create_date_list(date1, date2, delta)
    print(f"Pagerank con lag: {lag}\n")
    pagerank_df = compute_pagerank(df,delta,lag,dates,min_samples=min_samples)
    ut.save_dataframe(pagerank_df,"f:\\TFG\\datasets\\raw_datasets\\pagerank\\",f"pagerank_{delta}m_{lag}l_min{min_samples}")

    

Pagerank con lag: 1

Pagerank con lag: 2

Pagerank con lag: 5

Pagerank con lag: 10



Cruzamos con los datos origen para asignar el pagerank a cada equipo

In [213]:
delta, lag = deltas[0], lags[0]
res = ut.read_data(f"f:\\TFG\\datasets\\raw_datasets\\pagerank\\pagerank_{delta}m_{lag}l_min{min_samples}.csv",["Date_pagerank"])
res = res.rename(columns={"pagerank":f"pagerank_{lag}"}).drop(columns="Unnamed: 0")
print(res.columns)

for delta,lag in zip(deltas[1:],lags[1:]):
    p = ut.read_data(f"f:\\TFG\\datasets\\raw_datasets\\pagerank\\pagerank_{delta}m_{lag}l_min{min_samples}.csv",["Date_pagerank"])
    p = p.rename(columns={"pagerank":f"pagerank_{lag}"}).drop(columns="Unnamed: 0")
    res = res.merge(p,how='inner',on=["Div","Team","Date_pagerank"])

ut.save_dataframe(res,"f:\\TFG\\datasets\\raw_datasets\\pagerank\\",f"common_pagerank_min{min_samples}")

Index(['Div', 'Team', 'pagerank_1', 'Date_pagerank'], dtype='object')


Cruzamos ahora con los partidos para asignar el pagerank corresponiente a cada partido y equipo

In [214]:
# data_filt = df[df.Date>"2020-01-01"][df.Div=="SP1"].copy()
data_filt = df[df.Date>date1].copy()
print(data_filt.shape)
print("\n")
prior_delta = -1 # in order to optimize the get_prior_date() computation

for delta,lag in zip(deltas,lags):
    print(f"Loop {lag} - getting prior dates",end='\r')
    dates = create_date_list(date1, date2, delta)
    if prior_delta!=delta:
        date_aux = data_filt.Date.apply(lambda x: datetime.strptime(get_prior_date(dates,x),"%Y-%m-%d"))
        data_filt.loc[:,"Date_pagerank"] = date_aux.apply(lambda x: datetime.strftime(x,"%Y-%m-%d"))
    else: prior_delta=delta

    print(f"Loop {lag} - merging data",end='\r')
    page_rank_permonth = ut.read_data(f"f:\\TFG\\datasets\\raw_datasets\\pagerank\\pagerank_{delta}m_{lag}l.csv",["Date_pagerank"])
    page_rank_permonth = page_rank_permonth.drop(columns="Unnamed: 0")

    data_filt = data_filt.merge(page_rank_permonth.rename(columns={"pagerank":f"pagerank_{lag}_H","Team":"HomeTeam"}),
                                   left_on=["Div","Date_pagerank","HomeTeam"],right_on=["Div","Date_pagerank","HomeTeam"])
    data_filt = data_filt.merge(page_rank_permonth.rename(columns={"pagerank":f"pagerank_{lag}_A","Team":"AwayTeam"}),
                                      left_on=["Div","Date_pagerank","AwayTeam"],right_on=["Div","Date_pagerank","AwayTeam"])
    
# data_filt = data_filt[["matchId","pagerank_H","pagerank_A"]]
data_filt

(60005, 43)




,matchId,id_H,id_A,Div,div_order,Date,season,HomeTeam,AwayTeam,FTHG,...,id2,Date_pagerank,pagerank_1_H,pagerank_1_A,pagerank_2_H,pagerank_2_A,pagerank_5_H,pagerank_5_A,pagerank_10_H,pagerank_10_A
0,0,8,19,D1,0,2000-08-11,T00-01,Dortmund,Hansa Rostock,1,...,Hansa Rostock,2000-05-25,0.567654,0.488550,0.604490,0.600576,0.845053,0.622272,0.762707,0.632111
1,42,26,19,D1,42,2000-09-16,T00-01,Leverkusen,Hansa Rostock,1,...,Leverkusen,2000-05-25,0.764302,0.505554,0.800447,0.585959,0.746261,0.622272,0.794117,0.632111
2,24,12,19,D1,24,2000-09-06,T00-01,Ein Frankfurt,Hansa Rostock,4,...,Hansa Rostock,2000-05-25,0.353437,0.505554,0.495112,0.585959,0.655970,0.622272,0.631917,0.632111
3,23,8,29,D1,23,2000-09-06,T00-01,Dortmund,Munich 1860,2,...,Munich 1860,2000-05-25,0.644826,0.597033,0.527913,0.502340,0.845053,0.553025,0.762707,0.588495
4,39,12,29,D1,39,2000-09-16,T00-01,Ein Frankfurt,Munich 1860,1,...,Munich 1860,2000-05-25,0.353437,0.597033,0.495112,0.502340,0.655970,0.553025,0.631917,0.588495
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
49437,59992,308,242,SP2,11411,2000-05-28,T99-00,Toledo,Ath Madrid B,1.0,...,Toledo,2000-05-25,0.539170,0.858792,0.581696,0.774332,0.617723,0.757056,0.608314,0.734637
49438,59993,314,271,SP2,11412,2000-05-28,T99-00,Villarreal,Las Palmas,1.0,...,Villarreal,2000-05-25,0.568317,0.741082,0.600549,0.810692,0.624675,0.725429,0.673730,0.725667
49439,59997,242,284,SP2,11416,2000-06-04,T99-00,Ath Madrid B,Merida,0.0,...,Merida,2000-05-25,0.858792,0.549146,0.774332,0.672290,0.757056,0.708068,0.734637,0.745711
49440,60002,277,314,SP2,11421,2000-06-04,T99-00,Logrones,Villarreal,1.0,...,Villarreal,2000-05-25,0.590267,0.568317,0.739854,0.600549,0.711910,0.624675,0.688064,0.673730


In [216]:
data_filt.columns

Index(['matchId', 'id_H', 'id_A', 'Div', 'div_order', 'Date', 'season',
       'HomeTeam', 'AwayTeam', 'FTHG', 'FTAG', 'FTR', 'Attendance', 'Referee',
       'HS', 'AS', 'HST', 'AST', 'HF', 'AF', 'HC', 'AC', 'HY', 'AY', 'HR',
       'AR', 'HO', 'AO', 'HHW', 'AHW', 'B365H', 'B365D', 'B365A', 'MaxH',
       'MaxD', 'MaxA', 'AvgH', 'AvgD', 'AvgA', 'Points_H', 'Points_A', 'id1',
       'id2', 'Date_pagerank', 'pagerank_1_H', 'pagerank_1_A', 'pagerank_2_H',
       'pagerank_2_A', 'pagerank_5_H', 'pagerank_5_A', 'pagerank_10_H',
       'pagerank_10_A'],
      dtype='object')

In [217]:
pagerank_res = data_filt[["matchId","Div","season","Date","HomeTeam","AwayTeam","FTHG","FTAG","FTR",
                          *data_filt.columns[data_filt.columns.map(lambda c: c.startswith("pagerank"))]]]
pagerank_res

,matchId,Div,season,Date,HomeTeam,AwayTeam,FTHG,FTAG,FTR,pagerank_1_H,pagerank_1_A,pagerank_2_H,pagerank_2_A,pagerank_5_H,pagerank_5_A,pagerank_10_H,pagerank_10_A
0,0,D1,T00-01,2000-08-11,Dortmund,Hansa Rostock,1,0,H,0.567654,0.488550,0.604490,0.600576,0.845053,0.622272,0.762707,0.632111
1,42,D1,T00-01,2000-09-16,Leverkusen,Hansa Rostock,1,2,A,0.764302,0.505554,0.800447,0.585959,0.746261,0.622272,0.794117,0.632111
2,24,D1,T00-01,2000-09-06,Ein Frankfurt,Hansa Rostock,4,0,H,0.353437,0.505554,0.495112,0.585959,0.655970,0.622272,0.631917,0.632111
3,23,D1,T00-01,2000-09-06,Dortmund,Munich 1860,2,3,A,0.644826,0.597033,0.527913,0.502340,0.845053,0.553025,0.762707,0.588495
4,39,D1,T00-01,2000-09-16,Ein Frankfurt,Munich 1860,1,0,H,0.353437,0.597033,0.495112,0.502340,0.655970,0.553025,0.631917,0.588495
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
49437,59992,SP2,T99-00,2000-05-28,Toledo,Ath Madrid B,1.0,1.0,D,0.539170,0.858792,0.581696,0.774332,0.617723,0.757056,0.608314,0.734637
49438,59993,SP2,T99-00,2000-05-28,Villarreal,Las Palmas,1.0,1.0,D,0.568317,0.741082,0.600549,0.810692,0.624675,0.725429,0.673730,0.725667
49439,59997,SP2,T99-00,2000-06-04,Ath Madrid B,Merida,0.0,0.0,D,0.858792,0.549146,0.774332,0.672290,0.757056,0.708068,0.734637,0.745711
49440,60002,SP2,T99-00,2000-06-04,Logrones,Villarreal,1.0,0.0,H,0.590267,0.568317,0.739854,0.600549,0.711910,0.624675,0.688064,0.673730


In [218]:
pagerank_res.to_csv('f:\\TFG\\datasets\\raw_datasets\\page_rank_v2.2_metadata.csv',sep=';',decimal=',',index=False)

In [38]:
version = "0.1"
pagerank_res = ut.read_data(f"f:\\TFG\\datasets\\raw_datasets\\page_rank_v{version}_metadata.csv",["Date"])

In [39]:
pagerank_res = pagerank_res[["matchId"]+[c for c in pagerank_res if c.startswith("pagerank")]]
pagerank_res

,matchId,pagerank_750D_H,pagerank_750D_A
0,0,0.386095,0.457999
1,42,1.000000,0.457999
2,24,0.338939,0.457999
3,23,0.386095,0.282556
4,3,0.836187,0.282556
...,...,...,...
54188,58541,0.699055,0.559702
54189,58542,0.836493,0.698756
54190,58543,0.551170,0.756887
54191,58544,0.771768,0.530623


In [42]:
if float(version)<2:
    pagerank_res.columns = ['matchId','pagerank_H','pagerank_A']
pagerank_res

,matchId,pagerank_H,pagerank_A
0,0,0.386095,0.457999
1,42,1.000000,0.457999
2,24,0.338939,0.457999
3,23,0.386095,0.282556
4,3,0.836187,0.282556
...,...,...,...
54188,58541,0.699055,0.559702
54189,58542,0.836493,0.698756
54190,58543,0.551170,0.756887
54191,58544,0.771768,0.530623


In [43]:
f'f:\\TFG\\datasets\\raw_datasets\\page_rank_v{version}.csv'

'f:\\TFG\\datasets\\raw_datasets\\page_rank_v0.1.csv'

In [44]:
pagerank_res.to_csv(f'f:\\TFG\\datasets\\raw_datasets\\page_rank_v{version}.csv',sep=';',decimal=',',index=False)

# MATCH IMPORTANCE

In [7]:
# ALGORITHM:
#   1) Calcular acumulado de puntos por equipo-temporada
#   2) Calcular jornada de cada partido
#   2) Tomar los 5 primeros y 5 ultimos -> guardarlo en una lista
        # a) Una vez calculados los puntos acumulados, y por cada partido:
            #   calcular el rank de puntos agrupando por Div-jornada.
#   3) Aplicar la formula de (rank(k) - rank(i)) / np -> np (partidos jugados), rank(k)
        # a) Columna de np (por cada equipo-temporada)
        # b) Columna de rank(i) / np
        # c) Columna de rank(k) / np
        # d) Columna aplicando la formula restando b) y c)
        # e) Repetir para k={1,2,3,4,5,-5,-4,-3,-2,-1}

In [8]:
version = "v3"
shift = True
df = ut.read_data("F:\TFG\datasets\\raw_datasets\datalake_v3.csv")
df

,matchId,id_H,id_A,Div,div_order,Date,Weekday,season,HomeTeam,AwayTeam,...,AHW,B365H,B365D,B365A,MaxH,MaxD,MaxA,AvgH,AvgD,AvgA
0,0,8,19,D1,0,2000-08-11,4,T00-01,Dortmund,Hansa Rostock,...,0.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,1,2,20,D1,1,2000-08-12,5,T00-01,Bayern Munich,Hertha,...,0.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,2,15,35,D1,2,2000-08-12,5,T00-01,Freiburg,Stuttgart,...,0.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,3,17,29,D1,3,2000-08-12,5,T00-01,Hamburg,Munich 1860,...,0.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,4,23,4,D1,4,2000-08-12,5,T00-01,Kaiserslautern,Bochum,...,0.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
111410,111410,483,532,T1,9439,2000-05-21,6,T99-00,Ankaragucu,Trabzonspor,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
111411,111411,486,482,T1,9440,2000-05-21,6,T99-00,Antalyaspor,Altay,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
111412,111412,490,515,T1,9441,2000-05-21,6,T99-00,Bursaspor,Kocaelispor,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
111413,111413,497,488,T1,9442,2000-05-21,6,T99-00,Erzurumspor,Besiktas,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [9]:
df.dtypes

matchId                int64
id_H                   int64
id_A                   int64
Div                   object
div_order              int64
Date          datetime64[ns]
Weekday                int64
season                object
HomeTeam              object
AwayTeam              object
FTHG                  object
FTAG                  object
FTR                   object
Attendance           float64
Referee               object
HS                    object
AS                    object
HST                   object
AST                   object
HF                    object
AF                    object
HC                    object
AC                    object
HY                    object
AY                    object
HR                    object
AR                    object
HO                    object
AO                    object
HHW                   object
AHW                   object
B365H                float64
B365D                float64
B365A                float64
MaxH          

In [10]:
COLS_H      = ['matchId','Div','Date','season','id_H','HomeTeam','FTHG','FTAG']
COLS_A      = ['matchId','Div','Date','season','id_A','AwayTeam','FTAG','FTHG']
COLS_AUX    = ['matchId','Div','Date','season','idTeam','Team','FTG','FTG_rival']

data_split = ut.split_data_side(df,COLS_H,COLS_A,COLS_AUX)
data_split.loc[:,"rounds"] = 1
data_split

,matchId,Div,Date,season,idTeam,Team,FTG,FTG_rival,Side,Points,rounds
6408,6408,D1,1993-08-07,T93-94,2,Bayern Munich,3.0,1.0,0,3,1
6408,6408,D1,1993-08-07,T93-94,15,Freiburg,1.0,3.0,1,0,1
6409,6409,D1,1993-08-07,T93-94,8,Dortmund,2.0,1.0,0,3,1
6409,6409,D1,1993-08-07,T93-94,24,Karlsruhe,1.0,2.0,1,0,1
6410,6410,D1,1993-08-07,T93-94,10,Duisburg,2.0,2.0,0,1,1
...,...,...,...,...,...,...,...,...,...,...,...
101968,101968,SP2,2024-06-02,T23-24,311,Valladolid,1,2,1,0,1
101969,101969,SP2,2024-06-02,T23-24,315,Villarreal B,1,0,0,3,1
101969,101969,SP2,2024-06-02,T23-24,301,Santander,0,1,1,0,1
101970,101970,SP2,2024-06-02,T23-24,317,Zaragoza,1,1,0,1,1


In [11]:
dates_dict = data_split[["matchId","Date"]].drop_duplicates().set_index("matchId").to_dict()['Date']

In [12]:
rounds = data_split.sort_values("Date").groupby(["Div","idTeam","season"]).expanding().agg({"rounds":"sum","Points":"mean"}).reset_index()
rounds.columns = ["Div","idTeam","season","matchId","rounds","Points_mean"]
rounds.loc[:,"Date"] = rounds.matchId.map(dates_dict)
rounds.sort_values(['Date','matchId'])

,Div,idTeam,season,matchId,rounds,Points_mean,Date
1288,D1,2,T93-94,6408,1.0,3.000000,1993-08-07
6514,D1,15,T93-94,6408,1.0,0.000000,1993-08-07
3562,D1,8,T93-94,6409,1.0,3.000000,1993-08-07
10522,D1,24,T93-94,6409,1.0,0.000000,1993-08-07
3936,D1,10,T93-94,6410,1.0,1.000000,1993-08-07
...,...,...,...,...,...,...,...
201727,SP2,311,T23-24,101968,42.0,1.714286,2024-06-02
198757,SP2,301,T23-24,101969,42.0,1.523810,2024-06-02
202559,SP2,315,T23-24,101969,42.0,1.023810,2024-06-02
179489,SP2,234,T23-24,101970,42.0,1.214286,2024-06-02


In [13]:
rounds.loc[:,"rank"] = rounds.groupby(["Div","season","rounds"]).Points_mean.rank(method='first',ascending=False)
rounds

,Div,idTeam,season,matchId,rounds,Points_mean,Date,rank
0,D1,0,T06-07,1838,1.0,0.000000,2006-08-12,13.0
1,D1,0,T06-07,1846,2.0,0.000000,2006-08-19,16.0
2,D1,0,T06-07,1857,3.0,1.000000,2006-08-26,11.0
3,D1,0,T06-07,1864,4.0,1.500000,2006-09-16,7.0
4,D1,0,T06-07,1873,5.0,1.200000,2006-09-23,11.0
...,...,...,...,...,...,...,...,...
222825,T1,537,T96-97,110460,30.0,0.366667,1997-04-20,18.0
222826,T1,537,T96-97,110464,31.0,0.354839,1997-05-04,18.0
222827,T1,537,T96-97,110472,32.0,0.343750,1997-05-11,18.0
222828,T1,537,T96-97,110487,33.0,0.333333,1997-05-18,18.0


In [14]:
if shift:
    rounds_shifted = rounds.sort_values('Date').groupby(["season","Div","idTeam"])[['Points_mean','rank']].shift(1)
    rounds[['Points_mean','rank']] = rounds_shifted
    print("Shift")
else: 
    print("NO Shift")
rounds_shifted

Shift


,Points_mean,rank
16934,NaN,NaN
4818,NaN,NaN
5564,NaN,NaN
12524,NaN,NaN
3936,NaN,NaN
...,...,...
201727,1.756098,1.0
185871,1.439024,8.0
179489,1.219512,14.0
203815,1.219512,16.0


In [15]:
rounds

,Div,idTeam,season,matchId,rounds,Points_mean,Date,rank
0,D1,0,T06-07,1838,1.0,NaN,2006-08-12,NaN
1,D1,0,T06-07,1846,2.0,0.000000,2006-08-19,13.0
2,D1,0,T06-07,1857,3.0,0.000000,2006-08-26,16.0
3,D1,0,T06-07,1864,4.0,1.000000,2006-09-16,11.0
4,D1,0,T06-07,1873,5.0,1.500000,2006-09-23,7.0
...,...,...,...,...,...,...,...,...
222825,T1,537,T96-97,110460,30.0,0.379310,1997-04-20,18.0
222826,T1,537,T96-97,110464,31.0,0.366667,1997-05-04,18.0
222827,T1,537,T96-97,110472,32.0,0.354839,1997-05-11,18.0
222828,T1,537,T96-97,110487,33.0,0.343750,1997-05-18,18.0


In [16]:
rounds[rounds['idTeam']==317].iloc[50:100]

,Div,idTeam,season,matchId,rounds,Points_mean,Date,rank
203114,SP2,317,T08-09,52369,9.0,1.750000,2008-10-25,4.0
203115,SP2,317,T08-09,52378,10.0,1.888889,2008-11-01,3.0
203116,SP2,317,T08-09,52397,11.0,1.700000,2008-11-09,4.0
203117,SP2,317,T08-09,52401,12.0,1.636364,2008-11-15,5.0
203118,SP2,317,T08-09,52414,13.0,1.583333,2008-11-22,6.0
203119,SP2,317,T08-09,52430,14.0,1.692308,2008-11-30,4.0
203120,SP2,317,T08-09,52438,15.0,1.785714,2008-12-06,3.0
203121,SP2,317,T08-09,52444,16.0,1.733333,2008-12-13,4.0
203122,SP2,317,T08-09,52460,17.0,1.625000,2008-12-20,4.0
203123,SP2,317,T08-09,52463,18.0,1.705882,2009-01-03,4.0


In [17]:
rounds[(rounds.Div=="SP2") & (rounds.season=='T10-11') & (rounds.rounds==1)].sort_values("rank")

,Div,idTeam,season,matchId,rounds,Points_mean,Date,rank
179116,SP2,234,T10-11,53208,1.0,NaN,2010-08-29,NaN
179654,SP2,235,T10-11,53208,1.0,NaN,2010-08-29,NaN
181524,SP2,244,T10-11,53202,1.0,NaN,2010-08-28,NaN
181940,SP2,245,T10-11,53209,1.0,NaN,2010-08-29,NaN
182612,SP2,248,T10-11,53207,1.0,NaN,2010-08-28,NaN
183276,SP2,250,T10-11,53202,1.0,NaN,2010-08-28,NaN
184032,SP2,253,T10-11,53210,1.0,NaN,2010-08-29,NaN
185536,SP2,256,T10-11,53203,1.0,NaN,2010-08-28,NaN
187166,SP2,262,T10-11,53205,1.0,NaN,2010-08-28,NaN
187502,SP2,263,T10-11,53204,1.0,NaN,2010-08-28,NaN


In [18]:
data_split = data_split.drop(columns="rounds").merge(rounds, on=["Div","idTeam","season","matchId","Date"])
data_split

,matchId,Div,Date,season,idTeam,Team,FTG,FTG_rival,Side,Points,rounds,Points_mean,rank
0,6408,D1,1993-08-07,T93-94,2,Bayern Munich,3.0,1.0,0,3,1.0,NaN,NaN
1,6408,D1,1993-08-07,T93-94,15,Freiburg,1.0,3.0,1,0,1.0,NaN,NaN
2,6409,D1,1993-08-07,T93-94,8,Dortmund,2.0,1.0,0,3,1.0,NaN,NaN
3,6409,D1,1993-08-07,T93-94,24,Karlsruhe,1.0,2.0,1,0,1.0,NaN,NaN
4,6410,D1,1993-08-07,T93-94,10,Duisburg,2.0,2.0,0,1,1.0,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...
222825,101968,SP2,2024-06-02,T23-24,311,Valladolid,1,2,1,0,42.0,1.756098,1.0
222826,101969,SP2,2024-06-02,T23-24,315,Villarreal B,1,0,0,3,42.0,0.975610,21.0
222827,101969,SP2,2024-06-02,T23-24,301,Santander,0,1,1,0,42.0,1.560976,6.0
222828,101970,SP2,2024-06-02,T23-24,317,Zaragoza,1,1,0,1,42.0,1.219512,15.0


In [19]:
data_split[(data_split.Div=="SP2") & (data_split.season=='T10-11') & (data_split.rounds==2)].sort_values("rank")

,matchId,Div,Date,season,idTeam,Team,FTG,FTG_rival,Side,Points,rounds,Points_mean,rank
118786,53219,SP2,2010-09-05,T10-11,244,Barcelona B,2,1,0,3,2.0,3.0,1.0
118781,53216,SP2,2010-09-04,T10-11,245,Betis,3,1,1,3,2.0,3.0,2.0
118772,53212,SP2,2010-09-04,T10-11,248,Cartagena,0,1,0,0,2.0,3.0,3.0
118773,53212,SP2,2010-09-04,T10-11,256,Elche,1,0,1,3,2.0,3.0,4.0
118777,53214,SP2,2010-09-04,T10-11,263,Girona,0,2,1,0,2.0,3.0,5.0
118779,53215,SP2,2010-09-04,T10-11,271,Las Palmas,0,0,1,1,2.0,3.0,6.0
118789,53220,SP2,2010-09-05,T10-11,311,Valladolid,1,0,1,3,2.0,3.0,7.0
118775,53213,SP2,2010-09-04,T10-11,312,Vallecano,3,2,1,3,2.0,3.0,8.0
118783,53217,SP2,2010-09-04,T10-11,234,Albacete,0,1,1,0,2.0,1.0,9.0
118784,53218,SP2,2010-09-05,T10-11,235,Alcorcon,2,0,0,3,2.0,1.0,10.0


In [20]:
rounds = rounds[["Div","season","rounds","rank","Points_mean"]].dropna().groupby(['Div','season','rounds','rank']).Points_mean.mean().reset_index()
rounds

,Div,season,rounds,rank,Points_mean
0,D1,T00-01,2.0,1.0,3.000000
1,D1,T00-01,2.0,2.0,3.000000
2,D1,T00-01,2.0,3.0,3.000000
3,D1,T00-01,2.0,4.0,3.000000
4,D1,T00-01,2.0,5.0,3.000000
...,...,...,...,...,...
216906,T1,T99-00,34.0,14.0,1.151515
216907,T1,T99-00,34.0,15.0,1.121212
216908,T1,T99-00,34.0,16.0,1.090909
216909,T1,T99-00,34.0,17.0,0.696970


In [21]:
index_pivoting=['matchId', 'Div', 'Date', 'season','idTeam', 'Team', 'FTG',
       'FTG_rival', 'Side', 'Points', 'rounds', 'Points_mean', 'rank']

In [22]:
data_rounds = data_split.merge(rounds,on=["Div","season","rounds"],suffixes=("","_others")).dropna()
data_rounds

,matchId,Div,Date,season,idTeam,Team,FTG,FTG_rival,Side,Points,rounds,Points_mean,rank,rank_others,Points_mean_others
0,6417,D1,1993-08-14,T93-94,9,Dresden,0.0,1.0,0,0,2.0,1.000000,8.0,1.0,3.000000
1,6417,D1,1993-08-14,T93-94,9,Dresden,0.0,1.0,0,0,2.0,1.000000,8.0,2.0,3.000000
2,6417,D1,1993-08-14,T93-94,9,Dresden,0.0,1.0,0,0,2.0,1.000000,8.0,3.0,3.000000
3,6417,D1,1993-08-14,T93-94,9,Dresden,0.0,1.0,0,0,2.0,1.000000,8.0,4.0,3.000000
4,6417,D1,1993-08-14,T93-94,9,Dresden,0.0,1.0,0,0,2.0,1.000000,8.0,5.0,3.000000
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
4362030,101970,SP2,2024-06-02,T23-24,234,Albacete,1,1,1,1,42.0,1.219512,14.0,18.0,1.121951
4362031,101970,SP2,2024-06-02,T23-24,234,Albacete,1,1,1,1,42.0,1.219512,14.0,19.0,1.097561
4362032,101970,SP2,2024-06-02,T23-24,234,Albacete,1,1,1,1,42.0,1.219512,14.0,20.0,1.048780
4362033,101970,SP2,2024-06-02,T23-24,234,Albacete,1,1,1,1,42.0,1.219512,14.0,21.0,0.975610


In [23]:
data_rounds.columns

Index(['matchId', 'Div', 'Date', 'season', 'idTeam', 'Team', 'FTG',
       'FTG_rival', 'Side', 'Points', 'rounds', 'Points_mean', 'rank',
       'rank_others', 'Points_mean_others'],
      dtype='object')

In [24]:
data_rounds.pivot(index=index_pivoting,columns='rank_others',values='Points_mean_others')

rank_others                                                                                               1.0   \
matchId Div Date       season idTeam Team          FTG FTG_rival Side Points rounds Points_mean rank             
9       D1  2000-08-18 T00-01 6      Cottbus       1   4         0    0      2.0    0.000000    11.0  3.000000   
                              8      Dortmund      4   1         1    3      2.0    3.000000    3.0   3.000000   
10      D1  2000-08-19 T00-01 29     Munich 1860   2   1         0    3      2.0    1.000000    10.0  3.000000   
                              41     Werder Bremen 1   2         1    0      2.0    3.000000    8.0   3.000000   
11      D1  2000-08-19 T00-01 2      Bayern Munich 3   0         1    3      2.0    3.000000    1.0   3.000000   
...                                                                                                        ...   
111412  T1  2000-05-21 T99-00 515    Kocaelispor   0.0 2.0       1    0      34.0   1.212121    10.0  2.363636   
111413  T1  2000-05-21 T99-00 488    Besiktas      4.0 1.0       1    3      34.0   2.181818    2.0   2.363636   
                              497    Erzurumspor   1.0 4.0       0    0      34.0   1.151515    14.0  2.363636   
111414  T1  2000-05-21 T99-00 500    Galatasaray   1.0 1.0       0    1      34.0   2.363636    1.0   2.363636   
                              508    Istanbulspor  1.0 1.0       1    1      34.0   1.090909    16.0  2.363636   

rank_others                                                                                               2.0   \
matchId Div Date       season idTeam Team          FTG FTG_rival Side Points rounds Points_mean rank             
9       D1  2000-08-18 T00-01 6      Cottbus       1   4         0    0      2.0    0.000000    11.0  3.000000   
                              8      Dortmund      4   1         1    3      2.0    3.000000    3.0   3.000000   
10      D1  2000-08-19 T00-01 29     Munich 1860   2   1         0    3      2.0    1.000000    10.0  3.000000   
                              41     Werder Bremen 1   2         1    0      2.0    3.000000    8.0   3.000000   
11      D1  2000-08-19 T00-01 2      Bayern Munich 3   0         1    3      2.0    3.000000    1.0   3.000000   
...                                                                                                        ...   
111412  T1  2000-05-21 T99-00 515    Kocaelispor   0.0 2.0       1    0      34.0   1.212121    10.0  2.181818   
111413  T1  2000-05-21 T99-00 488    Besiktas      4.0 1.0       1    3      34.0   2.181818    2.0   2.181818   
                              497    Erzurumspor   1.0 4.0       0    0      34.0   1.151515    14.0  2.181818   
111414  T1  2000-05-21 T99-00 500    Galatasaray   1.0 1.0       0    1      34.0   2.363636    1.0   2.181818   
                              508    Istanbulspor  1.0 1.0       1    1      34.0   1.090909    16.0  2.181818   

rank_others                                                                                               3.0   \
matchId Div Date       season idTeam Team          FTG FTG_rival Side Points rounds Points_mean rank             
9       D1  2000-08-18 T00-01 6      Cottbus       1   4         0    0      2.0    0.000000    11.0  3.000000   
                              8      Dortmund      4   1         1    3      2.0    3.000000    3.0   3.000000   
10      D1  2000-08-19 T00-01 29     Munich 1860   2   1         0    3      2.0    1.000000    10.0  3.000000   
                              41     Werder Bremen 1   2         1    0      2.0    3.000000    8.0   3.000000   
11      D1  2000-08-19 T00-01 2      Bayern Munich 3   0         1    3      2.0    3.000000    1.0   3.000000   
...                                                                                                        ...   
111412  T1  2000-05-21 T99-00 515    Kocaelispor   0.0 2.0       1    0      34.0   1.212121    10.0  1.878788   
111413  

In [25]:
data_rounds = data_rounds.pivot(index=['matchId', 'Div', 'Date', 'season','idTeam', 'Team', 'FTG',
       'FTG_rival', 'Side', 'Points', 'rounds', 'Points_mean', 'rank'],columns='rank_others',values='Points_mean_others')
data_rounds = data_rounds.reset_index()
data_rounds

rank_others,matchId,Div,Date,season,idTeam,Team,FTG,FTG_rival,Side,Points,...,15.0,16.0,17.0,18.0,19.0,20.0,21.0,22.0,23.0,24.0
0,9,D1,2000-08-18,T00-01,6,Cottbus,1,4,0,0,...,0.000000,0.000000,0.00000,0.000000,NaN,NaN,NaN,NaN,NaN,NaN
1,9,D1,2000-08-18,T00-01,8,Dortmund,4,1,1,3,...,0.000000,0.000000,0.00000,0.000000,NaN,NaN,NaN,NaN,NaN,NaN
2,10,D1,2000-08-19,T00-01,29,Munich 1860,2,1,0,3,...,0.000000,0.000000,0.00000,0.000000,NaN,NaN,NaN,NaN,NaN,NaN
3,10,D1,2000-08-19,T00-01,41,Werder Bremen,1,2,1,0,...,0.000000,0.000000,0.00000,0.000000,NaN,NaN,NaN,NaN,NaN,NaN
4,11,D1,2000-08-19,T00-01,2,Bayern Munich,3,0,1,3,...,0.000000,0.000000,0.00000,0.000000,NaN,NaN,NaN,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
216906,111412,T1,2000-05-21,T99-00,515,Kocaelispor,0.0,2.0,1,0,...,1.121212,1.090909,0.69697,0.545455,NaN,NaN,NaN,NaN,NaN,NaN
216907,111413,T1,2000-05-21,T99-00,488,Besiktas,4.0,1.0,1,3,...,1.121212,1.090909,0.69697,0.545455,NaN,NaN,NaN,NaN,NaN,NaN
216908,111413,T1,2000-05-21,T99-00,497,Erzurumspor,1.0,4.0,0,0,...,1.121212,1.090909,0.69697,0.545455,NaN,NaN,NaN,NaN,NaN,NaN
216909,111414,T1,2000-05-21,T99-00,500,Galatasaray,1.0,1.0,0,1,...,1.121212,1.090909,0.69697,0.545455,NaN,NaN,NaN,NaN,NaN,NaN


In [26]:
data_rounds.columns

Index([    'matchId',         'Div',        'Date',      'season',
            'idTeam',        'Team',         'FTG',   'FTG_rival',
              'Side',      'Points',      'rounds', 'Points_mean',
              'rank',           1.0,           2.0,           3.0,
                 4.0,           5.0,           6.0,           7.0,
                 8.0,           9.0,          10.0,          11.0,
                12.0,          13.0,          14.0,          15.0,
                16.0,          17.0,          18.0,          19.0,
                20.0,          21.0,          22.0,          23.0,
                24.0],
      dtype='object', name='rank_others')

In [27]:
for c in range(1,23):
    data_rounds.loc[:,c] = data_rounds.loc[:,c] - data_rounds.loc[:,"Points_mean"] 

data_rounds

rank_others,matchId,Div,Date,season,idTeam,Team,FTG,FTG_rival,Side,Points,...,15.0,16.0,17.0,18.0,19.0,20.0,21.0,22.0,23.0,24.0
0,9,D1,2000-08-18,T00-01,6,Cottbus,1,4,0,0,...,0.000000,0.000000,0.000000,0.000000,NaN,NaN,NaN,NaN,NaN,NaN
1,9,D1,2000-08-18,T00-01,8,Dortmund,4,1,1,3,...,-3.000000,-3.000000,-3.000000,-3.000000,NaN,NaN,NaN,NaN,NaN,NaN
2,10,D1,2000-08-19,T00-01,29,Munich 1860,2,1,0,3,...,-1.000000,-1.000000,-1.000000,-1.000000,NaN,NaN,NaN,NaN,NaN,NaN
3,10,D1,2000-08-19,T00-01,41,Werder Bremen,1,2,1,0,...,-3.000000,-3.000000,-3.000000,-3.000000,NaN,NaN,NaN,NaN,NaN,NaN
4,11,D1,2000-08-19,T00-01,2,Bayern Munich,3,0,1,3,...,-3.000000,-3.000000,-3.000000,-3.000000,NaN,NaN,NaN,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
216906,111412,T1,2000-05-21,T99-00,515,Kocaelispor,0.0,2.0,1,0,...,-0.090909,-0.121212,-0.515152,-0.666667,NaN,NaN,NaN,NaN,NaN,NaN
216907,111413,T1,2000-05-21,T99-00,488,Besiktas,4.0,1.0,1,3,...,-1.060606,-1.090909,-1.484848,-1.636364,NaN,NaN,NaN,NaN,NaN,NaN
216908,111413,T1,2000-05-21,T99-00,497,Erzurumspor,1.0,4.0,0,0,...,-0.030303,-0.060606,-0.454545,-0.606061,NaN,NaN,NaN,NaN,NaN,NaN
216909,111414,T1,2000-05-21,T99-00,500,Galatasaray,1.0,1.0,0,1,...,-1.242424,-1.272727,-1.666667,-1.818182,NaN,NaN,NaN,NaN,NaN,NaN


In [28]:
data_rounds.loc[:,["Points_mean",1,2,3,8,12,16,20]]

rank_others,Points_mean,1.0,2.0,3.0,8.0,12.0,16.0,20.0
0,0.000000,3.000000,3.000000,3.000000,3.000000,0.000000,0.000000,NaN
1,3.000000,0.000000,0.000000,0.000000,0.000000,-3.000000,-3.000000,NaN
2,1.000000,2.000000,2.000000,2.000000,2.000000,-1.000000,-1.000000,NaN
3,3.000000,0.000000,0.000000,0.000000,0.000000,-3.000000,-3.000000,NaN
4,3.000000,0.000000,0.000000,0.000000,0.000000,-3.000000,-3.000000,NaN
...,...,...,...,...,...,...,...,...
216906,1.212121,1.151515,0.969697,0.666667,0.212121,-0.060606,-0.121212,NaN
216907,2.181818,0.181818,0.000000,-0.303030,-0.757576,-1.030303,-1.090909,NaN
216908,1.151515,1.212121,1.030303,0.727273,0.272727,0.000000,-0.060606,NaN
216909,2.363636,0.000000,-0.181818,-0.484848,-0.939394,-1.212121,-1.272727,NaN


In [29]:
match_importance = data_rounds[['matchId','Div','Date','season','idTeam','Team','FTG','FTG_rival','Side','Points','rounds','rank']].copy()
match_importance

rank_others,matchId,Div,Date,season,idTeam,Team,FTG,FTG_rival,Side,Points,rounds,rank
0,9,D1,2000-08-18,T00-01,6,Cottbus,1,4,0,0,2.0,11.0
1,9,D1,2000-08-18,T00-01,8,Dortmund,4,1,1,3,2.0,3.0
2,10,D1,2000-08-19,T00-01,29,Munich 1860,2,1,0,3,2.0,10.0
3,10,D1,2000-08-19,T00-01,41,Werder Bremen,1,2,1,0,2.0,8.0
4,11,D1,2000-08-19,T00-01,2,Bayern Munich,3,0,1,3,2.0,1.0
...,...,...,...,...,...,...,...,...,...,...,...,...
216906,111412,T1,2000-05-21,T99-00,515,Kocaelispor,0.0,2.0,1,0,34.0,10.0
216907,111413,T1,2000-05-21,T99-00,488,Besiktas,4.0,1.0,1,3,34.0,2.0
216908,111413,T1,2000-05-21,T99-00,497,Erzurumspor,1.0,4.0,0,0,34.0,14.0
216909,111414,T1,2000-05-21,T99-00,500,Galatasaray,1.0,1.0,0,1,34.0,1.0


In [33]:
match_importance[(match_importance.Div=="SP1") & (match_importance.season=='T23-24') & (match_importance.rounds==29)].sort_values("rank")

rank_others,matchId,Div,Date,season,idTeam,Team,FTG,FTG_rival,Side,Points,rounds,rank
195667,100486,SP1,2024-03-16,T23-24,219,Real Madrid,4,2,1,3,29.0,1.0
195669,100487,SP1,2024-03-16,T23-24,202,Girona,0,1,1,0,29.0,2.0
195681,100493,SP1,2024-03-17,T23-24,190,Barcelona,3,0,1,3,29.0,3.0
195680,100493,SP1,2024-03-17,T23-24,189,Ath Madrid,0,3,0,0,29.0,4.0
195671,100488,SP1,2024-03-16,T23-24,188,Ath Bilbao,2,0,0,3,29.0,5.0
195663,100484,SP1,2024-03-15,T23-24,224,Sociedad,2,0,0,3,29.0,6.0
195678,100492,SP1,2024-03-17,T23-24,191,Betis,0,2,1,0,29.0,7.0
195689,100497,SP1,2024-03-30,T23-24,227,Valencia,0,0,0,1,29.0,8.0
195675,100490,SP1,2024-03-17,T23-24,207,Las Palmas,0,1,0,0,29.0,9.0
195666,100486,SP1,2024-03-16,T23-24,217,Osasuna,2,4,0,0,29.0,10.0


In [34]:
# take top 5
match_importance.loc[:,"top1"] = data_rounds.loc[:,1]
match_importance.loc[:,"top2"] = data_rounds.loc[:,2]
match_importance.loc[:,"top3"] = data_rounds.loc[:,3]
match_importance.loc[:,"top4"] = data_rounds.loc[:,4]
match_importance.loc[:,"top5"] = data_rounds.loc[:,5]
match_importance.loc[:,"top6"] = data_rounds.loc[:,6]
match_importance.loc[:,"top7"] = data_rounds.loc[:,7]

match_importance

rank_others,matchId,Div,Date,season,idTeam,Team,FTG,FTG_rival,Side,Points,rounds,rank,top1,top2,top3,top4,top5,top6,top7
0,9,D1,2000-08-18,T00-01,6,Cottbus,1,4,0,0,2.0,11.0,3.000000,3.000000,3.000000,3.000000,3.000000,3.000000,3.000000
1,9,D1,2000-08-18,T00-01,8,Dortmund,4,1,1,3,2.0,3.0,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
2,10,D1,2000-08-19,T00-01,29,Munich 1860,2,1,0,3,2.0,10.0,2.000000,2.000000,2.000000,2.000000,2.000000,2.000000,2.000000
3,10,D1,2000-08-19,T00-01,41,Werder Bremen,1,2,1,0,2.0,8.0,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
4,11,D1,2000-08-19,T00-01,2,Bayern Munich,3,0,1,3,2.0,1.0,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
216906,111412,T1,2000-05-21,T99-00,515,Kocaelispor,0.0,2.0,1,0,34.0,10.0,1.151515,0.969697,0.666667,0.545455,0.393939,0.363636,0.363636
216907,111413,T1,2000-05-21,T99-00,488,Besiktas,4.0,1.0,1,3,34.0,2.0,0.181818,0.000000,-0.303030,-0.424242,-0.575758,-0.606061,-0.606061
216908,111413,T1,2000-05-21,T99-00,497,Erzurumspor,1.0,4.0,0,0,34.0,14.0,1.212121,1.030303,0.727273,0.606061,0.454545,0.424242,0.424242
216909,111414,T1,2000-05-21,T99-00,500,Galatasaray,1.0,1.0,0,1,34.0,1.0,0.000000,-0.181818,-0.484848,-0.606061,-0.757576,-0.787879,-0.787879


In [35]:
# take bottom 5

# rows w/ 22 teams

idx24 = data_rounds[24].isna().values
idx22 = data_rounds[22].isna().values
idx20 = data_rounds[20].isna().values
idx18 = data_rounds[18].isna().values
idx16 = data_rounds[16].isna().values

for pos,idx in zip([16,18,20,22,24],[idx16,idx18,idx20,idx22,idx24]):
    match_importance.loc[~idx,"down1"] = data_rounds.loc[~idx,pos]
    match_importance.loc[~idx,"down2"] = data_rounds.loc[~idx,pos-1]
    match_importance.loc[~idx,"down3"] = data_rounds.loc[~idx,pos-2]
    match_importance.loc[~idx,"down4"] = data_rounds.loc[~idx,pos-3]
    match_importance.loc[~idx,"down5"] = data_rounds.loc[~idx,pos-4]
    match_importance.loc[~idx,"_edited"] = pos

In [36]:
match_importance

rank_others,matchId,Div,Date,season,idTeam,Team,FTG,FTG_rival,Side,Points,...,top4,top5,top6,top7,down1,down2,down3,down4,down5,_edited
0,9,D1,2000-08-18,T00-01,6,Cottbus,1,4,0,0,...,3.000000,3.000000,3.000000,3.000000,0.000000,0.000000,0.000000,0.000000,0.000000,18.0
1,9,D1,2000-08-18,T00-01,8,Dortmund,4,1,1,3,...,0.000000,0.000000,0.000000,0.000000,-3.000000,-3.000000,-3.000000,-3.000000,-3.000000,18.0
2,10,D1,2000-08-19,T00-01,29,Munich 1860,2,1,0,3,...,2.000000,2.000000,2.000000,2.000000,-1.000000,-1.000000,-1.000000,-1.000000,-1.000000,18.0
3,10,D1,2000-08-19,T00-01,41,Werder Bremen,1,2,1,0,...,0.000000,0.000000,0.000000,0.000000,-3.000000,-3.000000,-3.000000,-3.000000,-3.000000,18.0
4,11,D1,2000-08-19,T00-01,2,Bayern Munich,3,0,1,3,...,0.000000,0.000000,0.000000,0.000000,-3.000000,-3.000000,-3.000000,-3.000000,-3.000000,18.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
216906,111412,T1,2000-05-21,T99-00,515,Kocaelispor,0.0,2.0,1,0,...,0.545455,0.393939,0.363636,0.363636,-0.666667,-0.515152,-0.121212,-0.090909,-0.060606,18.0
216907,111413,T1,2000-05-21,T99-00,488,Besiktas,4.0,1.0,1,3,...,-0.424242,-0.575758,-0.606061,-0.606061,-1.636364,-1.484848,-1.090909,-1.060606,-1.030303,18.0
216908,111413,T1,2000-05-21,T99-00,497,Erzurumspor,1.0,4.0,0,0,...,0.606061,0.454545,0.424242,0.424242,-0.606061,-0.454545,-0.060606,-0.030303,0.000000,18.0
216909,111414,T1,2000-05-21,T99-00,500,Galatasaray,1.0,1.0,0,1,...,-0.606061,-0.757576,-0.787879,-0.787879,-1.818182,-1.666667,-1.272727,-1.242424,-1.212121,18.0


In [39]:
match_importance[(match_importance.Div=="E1") & (match_importance.season=='T02-03') & (match_importance.rounds==45)].sort_values("rank")

rank_others,matchId,Div,Date,season,idTeam,Team,FTG,FTG_rival,Side,Points,...,top4,top5,top6,top7,down1,down2,down3,down4,down5,_edited
124034,63704,E1,2003-04-27,T02-03,365,Portsmouth,3,2,0,3,...,-0.363636,-0.409091,-0.454545,-0.568182,0.863636,0.909091,-1.159091,-1.022727,-0.977273,24.0
124032,63703,E1,2003-04-27,T02-03,350,Leicester,1,1,0,1,...,-0.318182,-0.363636,-0.409091,-0.522727,0.863636,0.909091,-1.113636,-0.977273,-0.931818,24.0
124037,63705,E1,2003-04-29,T02-03,371,Sheffield United,3,0,0,3,...,-0.022727,-0.068182,-0.113636,-0.227273,0.863636,0.909091,-0.818182,-0.681818,-0.636364,24.0
124038,63706,E1,2003-04-30,T02-03,368,Reading,3,0,1,3,...,0.000000,-0.045455,-0.090909,-0.204545,0.863636,0.909091,-0.795455,-0.659091,-0.613636,24.0
124029,63701,E1,2003-04-26,T02-03,387,Wolves,3,3,1,1,...,0.045455,0.000000,-0.045455,-0.159091,0.863636,0.909091,-0.750000,-0.613636,-0.568182,24.0
124023,63698,E1,2003-04-26,T02-03,358,Nott'm Forest,3,3,0,1,...,0.090909,0.045455,0.000000,-0.113636,0.863636,0.909091,-0.704545,-0.568182,-0.522727,24.0
124020,63697,E1,2003-04-26,T02-03,348,Ipswich,1,5,0,0,...,0.204545,0.159091,0.113636,0.000000,0.863636,0.909091,-0.590909,-0.454545,-0.409091,24.0
124033,63703,E1,2003-04-27,T02-03,357,Norwich,1,1,1,1,...,0.250000,0.204545,0.159091,0.045455,0.863636,0.909091,-0.545455,-0.409091,-0.363636,24.0
124022,63698,E1,2003-04-26,T02-03,354,Millwall,3,3,1,1,...,0.318182,0.272727,0.227273,0.113636,0.863636,0.909091,-0.477273,-0.340909,-0.295455,24.0
124021,63697,E1,2003-04-26,T02-03,386,Wimbledon,5,1,1,3,...,0.386364,0.340909,0.295455,0.181818,0.863636,0.909091,-0.409091,-0.272727,-0.227273,24.0


In [40]:
def merge_sides(df,group_on,col_order):
    mask_home = df.Side==0
    mask_away = df.Side==1
    data_merged = df[mask_home].merge(df[mask_away], on=group_on, suffixes=("_H","_A"))
    data_merged = data_merged[col_order]
    return data_merged

In [41]:
match_importance.columns

Index(['matchId', 'Div', 'Date', 'season', 'idTeam', 'Team', 'FTG',
       'FTG_rival', 'Side', 'Points', 'rounds', 'rank', 'top1', 'top2', 'top3',
       'top4', 'top5', 'top6', 'top7', 'down1', 'down2', 'down3', 'down4',
       'down5', '_edited'],
      dtype='object', name='rank_others')

In [42]:
group_on = ['matchId', 'Div', 'Date', 'season']
feats = [ [f+s for f in ['top1', 'top2', 'top3','top4', 'top5', 'top6', 'top7', 'down1', 'down2', 'down3', 'down4', 'down5']] for s in ['_H','_A']  ]
feats = [*feats[0],*feats[1]]
col_order = [*group_on, 'idTeam_H','idTeam_A','Team_H','Team_A',"rounds_H","rounds_A","rank_H","rank_A",*feats,"_edited_H","_edited_A"]
match_importance = merge_sides(match_importance.drop(columns=["FTG","FTG_rival"]),group_on,col_order)
match_importance

rank_others,matchId,Div,Date,season,idTeam_H,idTeam_A,Team_H,Team_A,rounds_H,rounds_A,...,top5_A,top6_A,top7_A,down1_A,down2_A,down3_A,down4_A,down5_A,_edited_H,_edited_A
0,9,D1,2000-08-18,T00-01,6,8,Cottbus,Dortmund,2.0,2.0,...,0.000000,0.000000,0.000000,-3.000000,-3.000000,-3.000000,-3.000000,-3.000000,18.0,18.0
1,10,D1,2000-08-19,T00-01,29,41,Munich 1860,Werder Bremen,2.0,2.0,...,0.000000,0.000000,0.000000,-3.000000,-3.000000,-3.000000,-3.000000,-3.000000,18.0,18.0
2,11,D1,2000-08-19,T00-01,4,2,Bochum,Bayern Munich,2.0,2.0,...,0.000000,0.000000,0.000000,-3.000000,-3.000000,-3.000000,-3.000000,-3.000000,18.0,18.0
3,12,D1,2000-08-19,T00-01,20,17,Hertha,Hamburg,2.0,2.0,...,2.000000,2.000000,2.000000,-1.000000,-1.000000,-1.000000,-1.000000,-1.000000,18.0,18.0
4,13,D1,2000-08-19,T00-01,35,26,Stuttgart,Leverkusen,2.0,2.0,...,0.000000,0.000000,0.000000,-3.000000,-3.000000,-3.000000,-3.000000,-3.000000,18.0,18.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
108420,111410,T1,2000-05-21,T99-00,483,532,Ankaragucu,Trabzonspor,34.0,34.0,...,0.030303,0.000000,0.000000,-1.030303,-0.878788,-0.484848,-0.454545,-0.424242,18.0,18.0
108421,111411,T1,2000-05-21,T99-00,486,482,Antalyaspor,Altay,34.0,34.0,...,0.484848,0.454545,0.454545,-0.575758,-0.424242,-0.030303,0.000000,0.030303,18.0,18.0
108422,111412,T1,2000-05-21,T99-00,490,515,Bursaspor,Kocaelispor,34.0,34.0,...,0.393939,0.363636,0.363636,-0.666667,-0.515152,-0.121212,-0.090909,-0.060606,18.0,18.0
108423,111413,T1,2000-05-21,T99-00,497,488,Erzurumspor,Besiktas,34.0,34.0,...,-0.575758,-0.606061,-0.606061,-1.636364,-1.484848,-1.090909,-1.060606,-1.030303,18.0,18.0


In [43]:
col_order = [*group_on, 'idTeam_H','idTeam_A','Team_H','Team_A',"rounds","rank_H","rank_A",*feats,"_edited_H","_edited_A"]

match_importance.loc[:,"rounds"] = match_importance.loc[:,["rounds_H","rounds_A"]].mean(axis=1)
match_importance = match_importance.drop(columns=["rounds_H","rounds_A"])
match_importance

rank_others,matchId,Div,Date,season,idTeam_H,idTeam_A,Team_H,Team_A,rank_H,rank_A,...,top6_A,top7_A,down1_A,down2_A,down3_A,down4_A,down5_A,_edited_H,_edited_A,rounds
0,9,D1,2000-08-18,T00-01,6,8,Cottbus,Dortmund,11.0,3.0,...,0.000000,0.000000,-3.000000,-3.000000,-3.000000,-3.000000,-3.000000,18.0,18.0,2.0
1,10,D1,2000-08-19,T00-01,29,41,Munich 1860,Werder Bremen,10.0,8.0,...,0.000000,0.000000,-3.000000,-3.000000,-3.000000,-3.000000,-3.000000,18.0,18.0,2.0
2,11,D1,2000-08-19,T00-01,4,2,Bochum,Bayern Munich,2.0,1.0,...,0.000000,0.000000,-3.000000,-3.000000,-3.000000,-3.000000,-3.000000,18.0,18.0,2.0
3,12,D1,2000-08-19,T00-01,20,17,Hertha,Hamburg,14.0,9.0,...,2.000000,2.000000,-1.000000,-1.000000,-1.000000,-1.000000,-1.000000,18.0,18.0,2.0
4,13,D1,2000-08-19,T00-01,35,26,Stuttgart,Leverkusen,16.0,6.0,...,0.000000,0.000000,-3.000000,-3.000000,-3.000000,-3.000000,-3.000000,18.0,18.0,2.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
108420,111410,T1,2000-05-21,T99-00,483,532,Ankaragucu,Trabzonspor,12.0,7.0,...,0.000000,0.000000,-1.030303,-0.878788,-0.484848,-0.454545,-0.424242,18.0,18.0,34.0
108421,111411,T1,2000-05-21,T99-00,486,482,Antalyaspor,Altay,13.0,15.0,...,0.454545,0.454545,-0.575758,-0.424242,-0.030303,0.000000,0.030303,18.0,18.0,34.0
108422,111412,T1,2000-05-21,T99-00,490,515,Bursaspor,Kocaelispor,11.0,10.0,...,0.363636,0.363636,-0.666667,-0.515152,-0.121212,-0.090909,-0.060606,18.0,18.0,34.0
108423,111413,T1,2000-05-21,T99-00,497,488,Erzurumspor,Besiktas,14.0,2.0,...,-0.606061,-0.606061,-1.636364,-1.484848,-1.090909,-1.060606,-1.030303,18.0,18.0,34.0


In [44]:
match_importance[match_importance.isnull().any(axis=1)]


rank_others,matchId,Div,Date,season,idTeam_H,idTeam_A,Team_H,Team_A,rank_H,rank_A,...,top6_A,top7_A,down1_A,down2_A,down3_A,down4_A,down5_A,_edited_H,_edited_A,rounds
16021,16487,E0,2021-05-07,T20-21,66,71,Leicester,Newcastle,3.0,17.0,...,NaN,0.588235,-0.558824,-0.294118,-0.264706,0.000000,0.000000,20.0,20.0,35.0
16022,16488,E0,2021-05-08,T20-21,65,85,Leeds,Tottenham,11.0,7.0,...,NaN,0.000000,-1.147059,-0.882353,-0.852941,-0.588235,-0.588235,20.0,20.0,35.0
16023,16489,E0,2021-05-08,T20-21,78,58,Sheffield United,Crystal Palace,20.0,13.0,...,0.484848,0.454545,-0.636364,-0.393939,-0.333333,-0.121212,-0.060606,20.0,20.0,34.5
16024,16490,E0,2021-05-08,T20-21,68,56,Man City,Chelsea,1.0,4.0,...,NaN,-0.147059,-1.294118,-1.029412,-1.000000,-0.735294,-0.735294,20.0,20.0,35.0
16026,16492,E0,2021-05-09,T20-21,91,52,Wolves,Brighton,12.0,14.0,...,NaN,0.558824,-0.588235,-0.323529,-0.294118,-0.029412,-0.029412,20.0,20.0,35.0
16028,16494,E0,2021-05-09,T20-21,88,60,West Ham,Everton,5.0,8.0,...,0.060606,0.030303,-1.060606,-0.818182,-0.757576,-0.545455,-0.484848,20.0,20.0,34.5
16029,16495,E0,2021-05-09,T20-21,43,87,Arsenal,West Brom,9.0,19.0,...,NaN,0.882353,-0.264706,0.000000,0.029412,0.294118,0.294118,20.0,20.0,35.0
16030,16496,E0,2021-05-10,T20-21,61,53,Fulham,Burnley,18.0,16.0,...,NaN,0.588235,-0.558824,-0.294118,-0.264706,0.000000,0.000000,20.0,20.0,35.0
16031,16497,E0,2021-05-11,T20-21,69,66,Man United,Leicester,2.0,4.0,...,NaN,-0.314286,NaN,NaN,NaN,NaN,NaN,20.0,NaN,35.5
16032,16498,E0,2021-05-11,T20-21,80,58,Southampton,Crystal Palace,15.0,13.0,...,NaN,0.441176,-0.705882,-0.441176,-0.411765,-0.147059,-0.147059,20.0,20.0,35.0


In [50]:
f'f:\\TFG\\datasets\\raw_datasets\\match_importance{version}.csv'

'f:\\TFG\\datasets\\raw_datasets\\match_importance_v3.csv'

In [51]:
match_importance[["matchId","rounds",*feats]].to_csv(f'f:\\TFG\\datasets\\raw_datasets\\match_importance{version}.csv',sep=';',decimal=',',index=False)
match_importance[["matchId","rounds",*feats]].dropna()

rank_others,matchId,rounds,top1_H,top2_H,top3_H,top4_H,top5_H,top6_H,top7_H,down1_H,...,top3_A,top4_A,top5_A,top6_A,top7_A,down1_A,down2_A,down3_A,down4_A,down5_A
0,9,2.0,3.000000,3.000000,3.000000,3.000000,3.000000,3.000000,3.000000,0.000000,...,0.000000,0.000000,0.000000,0.000000,0.000000,-3.000000,-3.000000,-3.000000,-3.000000,-3.000000
1,10,2.0,2.000000,2.000000,2.000000,2.000000,2.000000,2.000000,2.000000,-1.000000,...,0.000000,0.000000,0.000000,0.000000,0.000000,-3.000000,-3.000000,-3.000000,-3.000000,-3.000000
2,11,2.0,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,-3.000000,...,0.000000,0.000000,0.000000,0.000000,0.000000,-3.000000,-3.000000,-3.000000,-3.000000,-3.000000
3,12,2.0,3.000000,3.000000,3.000000,3.000000,3.000000,3.000000,3.000000,0.000000,...,2.000000,2.000000,2.000000,2.000000,2.000000,-1.000000,-1.000000,-1.000000,-1.000000,-1.000000
4,13,2.0,3.000000,3.000000,3.000000,3.000000,3.000000,3.000000,3.000000,0.000000,...,0.000000,0.000000,0.000000,0.000000,0.000000,-3.000000,-3.000000,-3.000000,-3.000000,-3.000000
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
108420,111410,34.0,1.212121,1.030303,0.727273,0.606061,0.454545,0.424242,0.424242,-0.606061,...,0.303030,0.181818,0.030303,0.000000,0.000000,-1.030303,-0.878788,-0.484848,-0.454545,-0.424242
108421,111411,34.0,1.212121,1.030303,0.727273,0.606061,0.454545,0.424242,0.424242,-0.606061,...,0.757576,0.636364,0.484848,0.454545,0.454545,-0.575758,-0.424242,-0.030303,0.000000,0.030303
108422,111412,34.0,1.181818,1.000000,0.696970,0.575758,0.424242,0.393939,0.393939,-0.636364,...,0.666667,0.545455,0.393939,0.363636,0.363636,-0.666667,-0.515152,-0.121212,-0.090909,-0.060606
108423,111413,34.0,1.212121,1.030303,0.727273,0.606061,0.454545,0.424242,0.424242,-0.606061,...,-0.303030,-0.424242,-0.575758,-0.606061,-0.606061,-1.636364,-1.484848,-1.090909,-1.060606,-1.030303


In [52]:
rounds[(rounds.Div=="D1") & (rounds.season=='T02-03') & (rounds.rounds==33)].sort_values("rank")

,Div,season,rounds,rank,Points_mean
1746,D1,T02-03,33.0,1.0,2.25000
1747,D1,T02-03,33.0,2.0,1.75000
1748,D1,T02-03,33.0,3.0,1.75000
1749,D1,T02-03,33.0,4.0,1.62500
1750,D1,T02-03,33.0,5.0,1.59375
1751,D1,T02-03,33.0,6.0,1.53125
1752,D1,T02-03,33.0,7.0,1.43750
1753,D1,T02-03,33.0,8.0,1.40625
1754,D1,T02-03,33.0,9.0,1.34375
1755,D1,T02-03,33.0,10.0,1.28125


In [53]:
df = match_importance
df[df.isna().any(axis=1)]

rank_others,matchId,Div,Date,season,idTeam_H,idTeam_A,Team_H,Team_A,rank_H,rank_A,...,top6_A,top7_A,down1_A,down2_A,down3_A,down4_A,down5_A,_edited_H,_edited_A,rounds
16021,16487,E0,2021-05-07,T20-21,66,71,Leicester,Newcastle,3.0,17.0,...,NaN,0.588235,-0.558824,-0.294118,-0.264706,0.000000,0.000000,20.0,20.0,35.0
16022,16488,E0,2021-05-08,T20-21,65,85,Leeds,Tottenham,11.0,7.0,...,NaN,0.000000,-1.147059,-0.882353,-0.852941,-0.588235,-0.588235,20.0,20.0,35.0
16023,16489,E0,2021-05-08,T20-21,78,58,Sheffield United,Crystal Palace,20.0,13.0,...,0.484848,0.454545,-0.636364,-0.393939,-0.333333,-0.121212,-0.060606,20.0,20.0,34.5
16024,16490,E0,2021-05-08,T20-21,68,56,Man City,Chelsea,1.0,4.0,...,NaN,-0.147059,-1.294118,-1.029412,-1.000000,-0.735294,-0.735294,20.0,20.0,35.0
16026,16492,E0,2021-05-09,T20-21,91,52,Wolves,Brighton,12.0,14.0,...,NaN,0.558824,-0.588235,-0.323529,-0.294118,-0.029412,-0.029412,20.0,20.0,35.0
16028,16494,E0,2021-05-09,T20-21,88,60,West Ham,Everton,5.0,8.0,...,0.060606,0.030303,-1.060606,-0.818182,-0.757576,-0.545455,-0.484848,20.0,20.0,34.5
16029,16495,E0,2021-05-09,T20-21,43,87,Arsenal,West Brom,9.0,19.0,...,NaN,0.882353,-0.264706,0.000000,0.029412,0.294118,0.294118,20.0,20.0,35.0
16030,16496,E0,2021-05-10,T20-21,61,53,Fulham,Burnley,18.0,16.0,...,NaN,0.588235,-0.558824,-0.294118,-0.264706,0.000000,0.000000,20.0,20.0,35.0
16031,16497,E0,2021-05-11,T20-21,69,66,Man United,Leicester,2.0,4.0,...,NaN,-0.314286,NaN,NaN,NaN,NaN,NaN,20.0,NaN,35.5
16032,16498,E0,2021-05-11,T20-21,80,58,Southampton,Crystal Palace,15.0,13.0,...,NaN,0.441176,-0.705882,-0.441176,-0.411765,-0.147059,-0.147059,20.0,20.0,35.0


# HISTORICAL STRENGTH

Remove the first 5 games of each season.

In [48]:
df

,matchId,id_H,id_A,Div,div_order,Date,season,HomeTeam,AwayTeam,FTHG,...,B365D,B365A,MaxH,MaxD,MaxA,AvgH,AvgD,AvgA,Points_H,Points_A
0,0,8,19,D1,0,2000-11-08,T00-01,Dortmund,Hansa Rostock,1,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,3,0
1,1,2,20,D1,1,2000-12-08,T00-01,Bayern Munich,Hertha,4,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,3,0
2,2,15,35,D1,2,2000-12-08,T00-01,Freiburg,Stuttgart,4,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,3,0
3,3,17,29,D1,3,2000-12-08,T00-01,Hamburg,Munich 1860,2,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,1,1
4,4,23,4,D1,4,2000-12-08,T00-01,Kaiserslautern,Bochum,0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0,3
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
58541,58541,258,243,SP2,10828,2000-04-06,T99-00,Extremadura,Badajoz,2.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,1,1
58542,58542,271,274,SP2,10829,2000-04-06,T99-00,Las Palmas,Levante,2.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,3,0
58543,58543,277,314,SP2,10830,2000-04-06,T99-00,Logrones,Villarreal,1.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,3,0
58544,58544,288,297,SP2,10831,2000-04-06,T99-00,Osasuna,Recreativo,2.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,3,0


In [49]:
old_data = df.copy()
ut.getPoints(df,"FTHG","FTAG","Points_H")
ut.getPoints(df,"FTAG","FTHG","Points_A")
df

,matchId,id_H,id_A,Div,div_order,Date,season,HomeTeam,AwayTeam,FTHG,...,B365D,B365A,MaxH,MaxD,MaxA,AvgH,AvgD,AvgA,Points_H,Points_A
0,0,8,19,D1,0,2000-11-08,T00-01,Dortmund,Hansa Rostock,1,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,3,0
1,1,2,20,D1,1,2000-12-08,T00-01,Bayern Munich,Hertha,4,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,3,0
2,2,15,35,D1,2,2000-12-08,T00-01,Freiburg,Stuttgart,4,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,3,0
3,3,17,29,D1,3,2000-12-08,T00-01,Hamburg,Munich 1860,2,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,1,1
4,4,23,4,D1,4,2000-12-08,T00-01,Kaiserslautern,Bochum,0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0,3
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
58541,58541,258,243,SP2,10828,2000-04-06,T99-00,Extremadura,Badajoz,2.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,1,1
58542,58542,271,274,SP2,10829,2000-04-06,T99-00,Las Palmas,Levante,2.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,3,0
58543,58543,277,314,SP2,10830,2000-04-06,T99-00,Logrones,Villarreal,1.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,3,0
58544,58544,288,297,SP2,10831,2000-04-06,T99-00,Osasuna,Recreativo,2.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,3,0


In [50]:
COLS_H      = ['matchId','Div','Date','season','id_H','HomeTeam','FTHG','FTAG']
COLS_A      = ['matchId','Div','Date','season','id_A','AwayTeam','FTAG','FTHG']
COLS_AUX    = ['matchId','Div','Date','season','idTeam','Team','FTG','FTG_rival']

data_split = ut.split_data_side(df,COLS_H,COLS_A,COLS_AUX)
data_split

f:\TFG\code\experiments\utils.py:44: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df_H.loc[:,'Side'] = 0
f:\TFG\code\experiments\utils.py:36: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df.loc[:,new_col] = 1
f:\TFG\code\experiments\utils.py:49: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returni

,matchId,Div,Date,season,idTeam,Team,FTG,FTG_rival,Side,Points
6316,6316,D1,1993-01-09,T93-94,2,Bayern Munich,3.0,0.0,0,3
6316,6316,D1,1993-01-09,T93-94,25,Leipzig,0.0,3.0,1,0
6317,6317,D1,1993-01-09,T93-94,8,Dortmund,4.0,0.0,0,3
6317,6317,D1,1993-01-09,T93-94,9,Dresden,0.0,4.0,1,0
6318,6318,D1,1993-01-09,T93-94,12,Ein Frankfurt,3.0,1.0,0,3
...,...,...,...,...,...,...,...,...,...,...
44863,44863,SP1,2021-12-05,T20-21,200,Getafe,0,1,1,0
44864,44864,SP1,2021-12-05,T20-21,205,Huesca,1,0,0,3
44864,44864,SP1,2021-12-05,T20-21,188,Ath Bilbao,0,1,1,0
44865,44865,SP1,2021-12-05,T20-21,189,Ath Madrid,2,1,0,3


In [51]:
pd.get_dummies(data_split.Points)

,0,1,3
6316,False,False,True
6316,True,False,False
6317,False,False,True
6317,True,False,False
6318,False,False,True
...,...,...,...
44863,True,False,False
44864,False,False,True
44864,True,False,False
44865,False,False,True


In [52]:
data_split[["Loss","Draw","Win"]] = pd.get_dummies(data_split.Points)
data_split

,matchId,Div,Date,season,idTeam,Team,FTG,FTG_rival,Side,Points,Loss,Draw,Win
6316,6316,D1,1993-01-09,T93-94,2,Bayern Munich,3.0,0.0,0,3,False,False,True
6316,6316,D1,1993-01-09,T93-94,25,Leipzig,0.0,3.0,1,0,True,False,False
6317,6317,D1,1993-01-09,T93-94,8,Dortmund,4.0,0.0,0,3,False,False,True
6317,6317,D1,1993-01-09,T93-94,9,Dresden,0.0,4.0,1,0,True,False,False
6318,6318,D1,1993-01-09,T93-94,12,Ein Frankfurt,3.0,1.0,0,3,False,False,True
...,...,...,...,...,...,...,...,...,...,...,...,...,...
44863,44863,SP1,2021-12-05,T20-21,200,Getafe,0,1,1,0,True,False,False
44864,44864,SP1,2021-12-05,T20-21,205,Huesca,1,0,0,3,False,False,True
44864,44864,SP1,2021-12-05,T20-21,188,Ath Bilbao,0,1,1,0,True,False,False
44865,44865,SP1,2021-12-05,T20-21,189,Ath Madrid,2,1,0,3,False,False,True


In [53]:
ut.compute_lags

<function utils.compute_lags(df, lags, min_samples, col_window, cols_group, aggregations, cols_agg=False)>

In [76]:
aggs = {"Draw":["mean"],"Win":["mean"],"FTG":["mean","std"],"FTG_rival":["mean","std"]}
cols_aggs = ["Draw","Win","FTG_mean","FTG_std","FTG_rival_mean","FTG_rival_std"]
long_term = ut.compute_lag(data_split,"730D",5,'Date',["idTeam","Team","Side"],aggs,cols_aggs)
long_term

Draw       Win  FTG_mean   FTG_std   
idTeam Team     Side Date                                                 
0      Aachen   0    2006-03-12       NaN       NaN       NaN       NaN  \
                     2006-04-11       NaN       NaN       NaN       NaN   
                     2006-08-19       NaN       NaN       NaN       NaN   
                     2006-09-16       NaN       NaN       NaN       NaN   
                     2006-09-30       NaN       NaN       NaN       NaN   
...                                   ...       ...       ...       ...   
317    Zaragoza 1    2021-04-30  0.261905  0.285714  1.023810  0.999710   
                     2021-05-04  0.285714  0.261905  1.023810  0.999710   
                     2021-07-02  0.285714  0.285714  1.023810  0.999710   
                     2021-12-02  0.275862  0.241379  0.827586  0.848064   
                     2021-12-03  0.300000  0.233333  0.833333  0.833908   

                                 FTG_rival_mean  FTG_rival_std  
idTeam Team     Side Date                                       
0      Aachen   0    2006-03-12             NaN            NaN  
                     2006-04-11             NaN            NaN  
                     2006-08-19             NaN            NaN  
                     2006-09-16             NaN            NaN  
                     2006-09-30             NaN            NaN  
...                                         ...            ...  
317    Zaragoza 1    2021-04-30        1.261905       0.989198  
                     2021-05-04        1.285714       0.994760  
                     2021-07-02        1.214286       0.976198  
                     2021-12-02        1.206897       0.861034  
                     2021-12-03        1.200000       0.846901  

[117092 rows x 6 columns]

In [65]:
aggs = {"Draw":["mean"],"Win":["mean"],"FTG":["mean","std"],"FTG_rival":["mean","std"]}
cols_aggs = ["Draw","Win","FTG_mean","FTG_std","FTG_rival_mean","FTG_rival_std"]
short_term = ut.compute_lag(data_split.sort_values("Date"),5,5,'Date',["idTeam","Team","season"],aggs,cols_aggs)
short_term

Draw  Win  FTG_mean   FTG_std   
idTeam Team     season Date                                        
0      Aachen   T07-08 2006-03-12   NaN  NaN       NaN       NaN  \
                       2006-04-11   NaN  NaN       NaN       NaN   
                       2006-07-11   NaN  NaN       NaN       NaN   
                       2006-08-19   NaN  NaN       NaN       NaN   
                       2006-08-26   NaN  NaN       NaN       NaN   
...                                 ...  ...       ...       ...   
317    Zaragoza T20-21 2021-08-01   0.4  0.6       1.2  0.836660   
                       2021-08-05   0.2  0.8       1.6  0.547723   
                       2021-11-04   0.2  0.8       1.2  0.836660   
                       2021-12-02   0.2  0.8       1.4  0.894427   
                       2021-12-03   0.4  0.6       1.4  0.894427   

                                   FTG_rival_mean  FTG_rival_std  
idTeam Team     season Date                                       
0      Aachen   T07-08 2006-03-12             NaN            NaN  
                       2006-04-11             NaN            NaN  
                       2006-07-11             NaN            NaN  
                       2006-08-19             NaN            NaN  
                       2006-08-26             NaN            NaN  
...                                           ...            ...  
317    Zaragoza T20-21 2021-08-01             0.6       0.894427  
                       2021-08-05             0.6       0.894427  
                       2021-11-04             0.2       0.447214  
                       2021-12-02             0.4       0.547723  
                       2021-12-03             0.6       0.547723  

[117092 rows x 6 columns]

In [66]:
feats = [ [f+s for f in cols_aggs] for s in ['_H','_A']  ]
feats = [*feats[0],*feats[1]]
feats

['Draw_H',
 'Win_H',
 'FTG_mean_H',
 'FTG_std_H',
 'FTG_rival_mean_H',
 'FTG_rival_std_H',
 'Draw_A',
 'Win_A',
 'FTG_mean_A',
 'FTG_std_A',
 'FTG_rival_mean_A',
 'FTG_rival_std_A']

In [67]:
data_split

,matchId,Div,Date,season,idTeam,Team,FTG,FTG_rival,Side,Points,Loss,Draw,Win
6316,6316,D1,1993-01-09,T93-94,2,Bayern Munich,3.0,0.0,0,3,False,False,True
6316,6316,D1,1993-01-09,T93-94,25,Leipzig,0.0,3.0,1,0,True,False,False
6317,6317,D1,1993-01-09,T93-94,8,Dortmund,4.0,0.0,0,3,False,False,True
6317,6317,D1,1993-01-09,T93-94,9,Dresden,0.0,4.0,1,0,True,False,False
6318,6318,D1,1993-01-09,T93-94,12,Ein Frankfurt,3.0,1.0,0,3,False,False,True
...,...,...,...,...,...,...,...,...,...,...,...,...,...
44863,44863,SP1,2021-12-05,T20-21,200,Getafe,0,1,1,0,True,False,False
44864,44864,SP1,2021-12-05,T20-21,205,Huesca,1,0,0,3,False,False,True
44864,44864,SP1,2021-12-05,T20-21,188,Ath Bilbao,0,1,1,0,True,False,False
44865,44865,SP1,2021-12-05,T20-21,189,Ath Madrid,2,1,0,3,False,False,True


In [68]:
short_term = short_term.reset_index()
short_term

,idTeam,Team,season,Date,Draw,Win,FTG_mean,FTG_std,FTG_rival_mean,FTG_rival_std
0,0,Aachen,T07-08,2006-03-12,NaN,NaN,NaN,NaN,NaN,NaN
1,0,Aachen,T07-08,2006-04-11,NaN,NaN,NaN,NaN,NaN,NaN
2,0,Aachen,T07-08,2006-07-11,NaN,NaN,NaN,NaN,NaN,NaN
3,0,Aachen,T07-08,2006-08-19,NaN,NaN,NaN,NaN,NaN,NaN
4,0,Aachen,T07-08,2006-08-26,NaN,NaN,NaN,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...
117087,317,Zaragoza,T20-21,2021-08-01,0.4,0.6,1.2,0.836660,0.6,0.894427
117088,317,Zaragoza,T20-21,2021-08-05,0.2,0.8,1.6,0.547723,0.6,0.894427
117089,317,Zaragoza,T20-21,2021-11-04,0.2,0.8,1.2,0.836660,0.2,0.447214
117090,317,Zaragoza,T20-21,2021-12-02,0.2,0.8,1.4,0.894427,0.4,0.547723


In [69]:
short_term = short_term.merge(data_split[['Date','idTeam','Side','matchId']],on=['Date','idTeam'])
short_term

,idTeam,Team,season,Date,Draw,Win,FTG_mean,FTG_std,FTG_rival_mean,FTG_rival_std,Side,matchId
0,0,Aachen,T07-08,2006-03-12,NaN,NaN,NaN,NaN,NaN,NaN,0,2147
1,0,Aachen,T07-08,2006-04-11,NaN,NaN,NaN,NaN,NaN,NaN,0,2096
2,0,Aachen,T07-08,2006-07-11,NaN,NaN,NaN,NaN,NaN,NaN,1,2104
3,0,Aachen,T07-08,2006-08-19,NaN,NaN,NaN,NaN,NaN,NaN,0,2024
4,0,Aachen,T07-08,2006-08-26,NaN,NaN,NaN,NaN,NaN,NaN,1,2035
...,...,...,...,...,...,...,...,...,...,...,...,...
117087,317,Zaragoza,T20-21,2021-08-01,0.4,0.6,1.2,0.836660,0.6,0.894427,0,56582
117088,317,Zaragoza,T20-21,2021-08-05,0.2,0.8,1.6,0.547723,0.6,0.894427,0,56771
117089,317,Zaragoza,T20-21,2021-11-04,0.2,0.8,1.2,0.836660,0.2,0.447214,0,56728
117090,317,Zaragoza,T20-21,2021-12-02,0.2,0.8,1.4,0.894427,0.4,0.547723,1,56625


In [70]:
def merge_sides(df,group_on,col_order):
    mask_home = df.Side==0
    mask_away = df.Side==1
    data_merged = df[mask_home].merge(df[mask_away], on=group_on, suffixes=("_H","_A"))
    data_merged = data_merged[col_order]
    return data_merged

In [75]:
merge_sides(short_term,["matchId"],col_order=["matchId",*feats])

,matchId,Draw_H,Win_H,FTG_mean_H,FTG_std_H,FTG_rival_mean_H,FTG_rival_std_H,Draw_A,Win_A,FTG_mean_A,FTG_std_A,FTG_rival_mean_A,FTG_rival_std_A
0,2147,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,2096,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,2024,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,2042,0.2,0.2,1.4,1.341641,1.6,1.816590,0.2,0.2,0.4,0.547723,1.0,0.707107
4,2060,0.2,0.4,1.6,1.816590,1.0,1.000000,0.2,0.2,0.8,0.836660,1.2,0.447214
...,...,...,...,...,...,...,...,...,...,...,...,...,...
58541,56747,0.2,0.2,0.4,0.547723,1.2,1.095445,0.4,0.2,0.4,0.547723,0.6,0.894427
58542,56660,0.6,0.2,0.8,0.836660,1.2,1.303840,1.0,0.0,0.6,0.547723,0.6,0.547723
58543,56582,0.4,0.6,1.2,0.836660,0.6,0.894427,0.6,0.0,0.6,0.894427,1.6,1.516575
58544,56771,0.2,0.8,1.6,0.547723,0.6,0.894427,0.6,0.4,2.0,1.414214,0.6,0.547723


In [81]:
long_term = long_term.reset_index().merge(data_split[['Date','idTeam','matchId']],on=['Date','idTeam'])
long_term

,idTeam,Team,Side,Date,Draw,Win,FTG_mean,FTG_std,FTG_rival_mean,FTG_rival_std,matchId
0,0,Aachen,0,2006-03-12,NaN,NaN,NaN,NaN,NaN,NaN,2147
1,0,Aachen,0,2006-04-11,NaN,NaN,NaN,NaN,NaN,NaN,2096
2,0,Aachen,0,2006-08-19,NaN,NaN,NaN,NaN,NaN,NaN,2024
3,0,Aachen,0,2006-09-16,NaN,NaN,NaN,NaN,NaN,NaN,2042
4,0,Aachen,0,2006-09-30,NaN,NaN,NaN,NaN,NaN,NaN,2060
...,...,...,...,...,...,...,...,...,...,...,...
117087,317,Zaragoza,1,2021-04-30,0.261905,0.285714,1.023810,0.999710,1.261905,0.989198,56758
117088,317,Zaragoza,1,2021-05-04,0.285714,0.261905,1.023810,0.999710,1.285714,0.994760,56722
117089,317,Zaragoza,1,2021-07-02,0.285714,0.285714,1.023810,0.999710,1.214286,0.976198,56619
117090,317,Zaragoza,1,2021-12-02,0.275862,0.241379,0.827586,0.848064,1.206897,0.861034,56625


In [82]:
merge_sides(long_term,["matchId"],col_order=["matchId",*feats])

,matchId,Draw_H,Win_H,FTG_mean_H,FTG_std_H,FTG_rival_mean_H,FTG_rival_std_H,Draw_A,Win_A,FTG_mean_A,FTG_std_A,FTG_rival_mean_A,FTG_rival_std_A
0,2147,NaN,NaN,NaN,NaN,NaN,NaN,0.133333,0.200000,1.066667,0.961150,2.066667,1.387015
1,2096,NaN,NaN,NaN,NaN,NaN,NaN,0.388889,0.305556,1.000000,0.894427,1.138889,0.990030
2,2024,NaN,NaN,NaN,NaN,NaN,NaN,0.235294,0.411765,1.117647,1.007989,1.147059,1.158169
3,2042,NaN,NaN,NaN,NaN,NaN,NaN,0.290323,0.096774,0.741935,0.773207,1.677419,1.275071
4,2060,NaN,NaN,NaN,NaN,NaN,NaN,0.133333,0.200000,0.733333,0.798809,1.933333,1.334523
...,...,...,...,...,...,...,...,...,...,...,...,...,...
58541,56747,0.235294,0.411765,1.205882,1.066839,1.117647,1.225109,0.307692,0.256410,0.641026,0.742938,0.948718,0.971941
58542,56660,0.264706,0.382353,1.088235,0.965076,1.058824,1.229466,0.230769,0.230769,0.871795,1.173826,1.128205,0.893823
58543,56582,0.264706,0.411765,1.117647,0.945955,1.000000,1.230915,0.411765,0.176471,0.647059,0.861770,1.294118,1.311712
58544,56771,0.257143,0.428571,1.142857,0.943799,0.971429,1.224402,0.333333,0.444444,1.444444,1.199128,0.777778,0.646762


# ENSAMBLE

In [15]:
import dataflow_pi_rating as pirates

def get_labels(df,col,new_col):
    df.loc[:,new_col] = df.loc[:,col]
    return ut._create_label(df,new_col)

#### end